# v14 — 집품 후 고객취소 10% 및 실물 FIFO 편입

기존 v13 집품 오류·시간·검색 규칙 유지.


# 시뮬레이션 진행 단계 (v14)

STEP 1~3 가정/오류 분리 → STEP 4 도착순 FIFO → STEP 5~11 수요 대체 및 운영취소 → STEP 11-D/E 민감도 → STEP 12 경제가치 확장 → STEP 14~16 교육 아이디어 → 마지막 무결성 검증.

## v14 기준 결과

집품 후 고객취소: 525개, 실물 FIFO 슬롯 2,104개/시나리오.
대체·복구 수요 2,141개 기준 수동/자동 대체율 47.92%/54.18%(+6.26%p), 운영취소 확정 977/826개(15.46% 감소).
가정형 시뮬레이션이며 실제 도입 성과가 아님.


In [1]:
# 로컬/Colab 공통: 설치 여부를 묻지 않고 사용 가능한 한글 폰트를 선택합니다.
import matplotlib
from matplotlib import font_manager
font_candidates = ['NanumGothic', 'NanumSquare', 'UnDotum', 'Malgun Gothic', 'AppleGothic', 'Noto Sans CJK KR']
installed = {f.name for f in font_manager.fontManager.ttflist}
for _font in font_candidates:
    if _font in installed:
        matplotlib.rcParams['font.family'] = _font
        break
matplotlib.rcParams['axes.unicode_minus'] = False
# Jupyter Notebook에서 차트가 실제로 표시되도록 inline 백엔드 사용
try:
    from IPython import get_ipython
    if get_ipython() is not None:
        matplotlib.use('module://matplotlib_inline.backend_inline')
except ImportError:
    pass
print('그래프 폰트:', matplotlib.rcParams['font.family'])


그래프 폰트: ['NanumGothic']


## **STEP1 - 라이브러리 & 파라미터**

In [2]:
# =============================================
# STEP 1: 라이브러리 & 파라미터
# =============================================
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import matplotlib.pyplot as plt
import matplotlib
# 한글 폰트는 상단 환경설정 셀에서 자동 선택

# 랜덤 시드 고정
np.random.seed(42)
random.seed(42)

# ── 시뮬레이션 규모 설정 ──────────────────────────
N_ORDERS    = 150000
CANCEL_RATE = 0.05   # 운영 문제 후보+고객취소 표본 비율, 실제 확정 취소율 아님

# ── 고객취소 / 운영 문제 후보 사건 표본 비율 ──────────
CANCEL_TYPE_WEIGHTS = {
    '단순취소': 0.70,
    '운영취소': 0.30,
}

# 집품 후 고객취소 공급원 규모: 전체 고객취소의 10%
# 운영취소 KPI에는 포함하지 않고, 기존 고객취소 중 일부를 재고 공급 이벤트로만 분리한다.
POST_PICK_CUSTOMER_CANCEL_RATIO = 0.10

# ── 운영취소 내 사유 비율 ─────────────────────
# 전체 취소 대비: 오집 16.5% / 미집 7.5% / 집품파손 1.5% / 리빈파손 1.5% / 토트분실 3%
CANCEL_REASON_WEIGHTS = {
    '오집':     0.55,
    '미집':     0.25,
    '집품파손': 0.05,
    '리빈파손': 0.05,
    '토트분실': 0.10,
}

# ── 작업자 구성 ───────────────────────────────
# 오집/미집인 경우에만 작업자 유형 부여
WORKER_EXPERIENCED_RATIO = 0.67
WORKER_NOVICE_RATIO      = 1 - WORKER_EXPERIENCED_RATIO

# ── 카테고리 ─────────────────────────────────                       ** 이부분 제외 **
CATEGORIES       = ['채소', '과일', '가공식품', '우유', '축산']
CATEGORY_WEIGHTS = [0.20, 0.20, 0.30, 0.15, 0.15]
# TODO: 실제 카테고리별 취급 비중으로 교체

# ── 재사용 시간 기준 / 수동 탐색 확률 ─────────────────────
# [변경 메모]
# 1) 동일 product_id + sku_id의 검수 통과 후보 중, 이미 가용 상태가 된 '당일 미사용 재고'는
#    과거 30분 제한 없이 재사용 후보로 인정한다.
# 2) 즉시 사용할 당일 재고가 없을 때만, 취소 시점 이후 +30분 이내 도착하는 동일 SKU 후보를 기다린다.
#    → 미래 후보에만 30분 대기 한도를 적용한다.
# 3) 개선 전(수동)에만 처리 지연이 길어질수록 직접 탐색을 포기하고
#    재집을 선택할 가능성을 확률로 반영한다.
# 4) 개선 후(자동화)는 시스템 검색으로 대상 상품을 찾을 수 있으므로
#    공통 재고/대기 조건을 만족하면 추가 확률 없이 재사용한다.
MATCH_WINDOW_MIN = 30  # 미래 후보 최대 대기시간(분)
DELAY_THRESHOLD_MIN = 30
MANUAL_DELAYED_REUSE_ATTEMPT_PROB_30_TO_70 = 0.70  # 기존 기준 유지: 30분 초과~70분 이하 재사용 시도확률
MANUAL_DELAYED_REUSE_ATTEMPT_PROB_OVER_70 = 0.15   # 신규 비교군: 70분 초과 장기 지연 시 재사용 시도확률

# ── 시뮬레이션 기간 ──────────────────────────          시뮬용 24시간 데이터 생성
START_DATE = datetime(2024, 1, 1, 8, 0, 0)
SIM_DAYS   = 1

# ============================================================
# [수정] 처리 방식별 처리 속도 파라미터
# - 기존(수동)  : 작업자가 직접 취소 처리 → 60분당 약 200건
# - 자동화      : 프로그램이 취소 처리    → 60분당 약 420건
# ============================================================
MANUAL_THROUGHPUT    = 200   # 수동 처리: 60분당 처리 건수
AUTOMATED_THROUGHPUT = 420   # 자동화 처리: 60분당 처리 건수

# 건당 처리 소요 시간 (분)
MANUAL_TIME_PER_CASE    = round(60 / MANUAL_THROUGHPUT, 2)     # = 0.3분/건
AUTOMATED_TIME_PER_CASE = 60 / AUTOMATED_THROUGHPUT  # 420건/h를 정확히 반영 (약 0.142857분)  # ≒ 0.14분/건

# ============================================================
# 기존(수동) 처리 시 탐색 현장 검증 추가 딜레이
# - 검수(inspection, 90%)는 수동/자동화 공통으로 적용
# - 수동은 검수와 별개로, 탐색 시 작업자가 현장에서
#   상품 위치·상태를 직접 확인하는 시간이 추가로 발생
# - 이 현장 확인 시간을 search_time_min 에 추가 가중
# ============================================================
MANUAL_ONSITE_VERIFY_RANGE = (0.5, 2.0)                                                # 현장 검증 추가 소요 시간 범위 (분)

# ============================================================
# [수정] 탐색 대기 시간 파라미터
# - v9: 임의 탐색대기 10~60분을 KPI에서 제거하고 취소존 도착 FIFO 처리 큐 대기를 분리 계측
# - 취소존 실물에 대한 전산/현장 검색은 실제 FIFO 처리 완료 이후에만 시작
# - 자동화: 전산 직접검색(위치 정보 있는 경우) / 수동: 실물검색 누락률 10% 유지
# ============================================================
MANUAL_SEARCH_WAIT_RANGE = (10, 60)  # 과거 버전 호환용; v9 실측 KPI에는 사용하지 않음
AUTOMATED_SEARCH_WAIT    = 0         # 자동화: 즉시 탐색 (대기 없음)

# 개선 전 수동 탐색 매칭 누락률
# 동일 SKU 검수통과 후보가 있어도 수동으로 직접 찾는 과정에서 10%는 찾지 못하고 재집한다고 가정
# 개선 후 자동화에는 적용하지 않음
MANUAL_MATCH_MISS_PROB = 0.10

# 취소존 시간 흐름 (모두 가상 가정값)
LINE_TRANSIT_MIN = (5.0, 10.0)        # 집품 완료 → 담당 라인 정상 도착
CANCEL_ZONE_TRANSIT_MIN = (5.0, 10.0) # 오류 확인 → 취소존 도착 (일반 운영 오류)
LOST_TOTE_TO_ZONE_MIN = 5.0          # 지연 토트가 라인에 도착 → 취소존 도착
# 토트분실 물리적 회수 분포 (실측값 아님, 세 갈래 합계 100%)
# 90%: 발생 후 30분 이내 정상 회수, 관리자 대기 20/30분 이내 검색 성공 시 원주문 복구.
#  9%: 30분 내 확보 실패. 취소처리 후 1~15분 지연되어 당일 SKU 공급원으로 들어옴.
#  1%: 극히 드문 이송/공정 오류. 취소처리 후 2~4일 뒤 발견; 당일 공급원에서 제외.
# 모든 유형에서 원래 상품을 확보하지 못했더라도 기존 정상 재고의 동일 SKU 대체 가능.
# 토트분실의 80%는 이미 존재하는 WMS 토트 경로 조회로 조치 가능하다는 가정.
# 이 비율은 수동·자동 시나리오에 똑같이 적용(신규 검색툴 성과로 집계하지 않음).
EXISTING_TOTE_ROUTE_TRACE_PROB = 0.80
# 기존 조회 후 도착 상품을 원 라인에서 확인하는 시간: 0.3분 (실측 아님).
EXISTING_ROUTE_VERIFY_MIN = 0.30
TOTE_RETURN_WITHIN_30_PROB = 0.90
TOTE_MULTI_DAY_RECOVERY_PROB = 0.01
TOTE_LATE_SAME_DAY_RETURN_DELAY_RANGE = (1.0,15.0)
TOTE_MULTI_DAY_RECOVERY_LAG_DAYS = (2,4)
# 한 토트에 배치된 집품 단위(상품 유닛) 수: 사용자 지정 범위.
TOTE_CAPACITY_MIN, TOTE_CAPACITY_MAX = 10,20



# ============================================================
# [연간 절감액 확장용 가정 파라미터]
# ※ 아래 수치는 시뮬레이션용 가정값이며 실제 운영 데이터 확보 시 교체
# ============================================================

# 상품군별 주문 시점 기준 잔여 판매 가능기간 범위(일)
# - 상온 상품 기준 가정
# - sku_id는 동일 product_id 내부에서 유통기한을 구분하기 위한 ID
# - 동일 product_id + 동일 sku_id 조합은 동일 expiry_date 사용
SHELF_LIFE_RANGE_DAYS = {
    '채소':     (5, 14),
    '과일':     (5, 14),
    '가공식품': (30, 90),
    '우유':     (5, 14),
    '축산':     (3, 10),
}

# 상품군별 정상 판매가 범위(원)
# 정상 판매가는 product_id(상품 바코드) 기준으로 1회 생성하여 고정
PRICE_RANGE = {
    '채소':     (1500, 8000),
    '과일':     (2000, 15000),
    '가공식품': (1000, 20000),
    '우유':     (2000, 6000),
    '축산':     (5000, 30000),
}

# ------------------------------------------------------------
# 재진열 운영 기준
# - 오전조/오후조에서 처리된 취소상품은 원칙적으로 익일 12~17시 재진열
# - 오전조 취소건은 약 95%가 당일 재진열되지 않는다는 운영 경험을 반영
#   → 5%만 당일 21~24시 재진열 가능, 나머지는 익일 12~17시
# - 21~24시 재진열은 늦은 시간으로, 실제 판매기회는 익일로 넘어간다고
#   보수적으로 가정하여 경제가치 계산 시 판매기준일을 +1일 처리
# - 전일 오전조 → 전일 오후조 → 금일 오전조 순의 재진열 우선순위는
#   실제 진열대 잔여량/처리용량 데이터가 없으므로 현재는 별도 용량제약으로
#   강제하지 않고, 추후 실데이터 확보 시 큐/용량 모형으로 확장 가능
# ------------------------------------------------------------
MORNING_SHIFT_END_HOUR = 17
EVENING_SHIFT_END_HOUR = 2
MORNING_SAME_DAY_REDISPLAY_PROB = 0.05  # 금일 오전조 취소건 중 당일 재진열 가능 비율
REDISPLAY_AFTERNOON_START_HOUR = 12
REDISPLAY_AFTERNOON_END_HOUR = 17
REDISPLAY_LATE_START_HOUR = 21
REDISPLAY_LATE_END_HOUR = 24

# 재진열 대상 사유
REDISPLAY_ELIGIBLE_REASONS = ['오집', '토트분실', '고객취소_집품후']  # 실물 회수 공급원만

# ------------------------------------------------------------
# 상품군별 재진열 할인 규칙
# 형식: (잔여 유통기한 하한, 할인율), 위에서부터 큰 구간 순서로 판정
# ※ 실제 할인 정책 데이터가 없는 상태의 시뮬레이션 가정값
# ------------------------------------------------------------
DISCOUNT_RULES_BY_CATEGORY = {
    '채소': [
        (5, 0.00),
        (3, 0.10),
        (1, 0.20),
        (0, 0.30),
    ],
    '과일': [
        (7, 0.00),
        (4, 0.10),
        (2, 0.20),
        (0, 0.30),
    ],
    '가공식품': [
        (30, 0.00),
        (15, 0.10),
        (7, 0.20),
        (0, 0.30),
    ],
    '우유': [
        (7, 0.00),
        (4, 0.10),
        (2, 0.20),
        (0, 0.30),
    ],
    '축산': [
        (5, 0.00),
        (3, 0.10),
        (1, 0.20),
        (0, 0.30),
    ],
}

# 판매실패확률 가정은 제거함.
# 경제효과는 '회수된 정상 실물의 재진열 지연으로 추가로 발생하는
# 유통기한 할인·폐기 감가'만 측정하고, 정상 재고 대체의 기회비용·순이익을 계산하지 않는다.

# 의사결정 지원 도구의 검색 수행률 (관리자가 도구를 사용할 것으로 가정한 상한 시나리오)
BASE_TOOL_USAGE_RATE = 1.00
OP_REASONS = ['오집', '미집', '집품파손', '리빈파손', '토트분실']
SUPPLY_REASONS = ['오집', '토트분실', '고객취소_집품후']

# v9: 토트분실 인지시각(정상 라인 예상 도착시간)으로부터 관리자 대기 한도 적용
TOTE_WAIT_NORMAL_MIN = 30
TOTE_WAIT_PEAK_MIN = 20
PEAK_HOURS = (11, 12, 20, 21, 22, 23)
AUTOMATED_TOTE_DETECTION_MIN = 0.3  # 전산 입고/위치 등록 후 검색·확인 시간(가정)
AUTOMATED_LOCATION_KNOWN_PROB = 0.95


# v11: 기존 오집 표본 안에서만 과집/타토트 오집을 분기. 현장 비율 미확정.
OVERPICK_SHARE_OF_WRONG_PICK = 0.10  # 과집 10%, 타토트 오집 90%
PICK_SUPPLY_ONLY_REASONS = ['과집', '타토트오집_공급']
SUPPLY_REASONS = ['과집', '타토트오집_공급', '토트분실', '고객취소_집품후']
REDISPLAY_ELIGIBLE_REASONS = SUPPLY_REASONS.copy()


## **STEP2 - 주문 데이터 생성**

In [3]:
# =============================================
# STEP 2: 전체 주문 데이터 생성
# =============================================

# 셀 단위 시드 고정
random.seed(42)
np.random.seed(42)

# ============================================================
# [변경 메모]
# - product_id = 상품 바코드(동일 상품 식별)
# - sku_id = 동일 상품 내부의 유통기한 구분 표시용 ID
# - 대체 가능 상품 = product_id + sku_id가 모두 동일한 경우
# - 원본의 주문 생성 난수 흐름은 최대한 보존
# - 동일 product_id에 서로 다른 category가 붙는 문제만 최소 수정:
#   원본 주문에서 해당 product_id가 처음 관측된 category를 대표 category로 고정
# ============================================================

def generate_orders(n):
    order_times = []
    for _ in range(n):
        day    = random.randint(0, SIM_DAYS - 1)
        hour   = random.choices(
            range(8, 24),
            weights=[3,4,6,8,8,7,6,7,8,9,8,7,6,5,4,3],
            k=1
        )[0]
        minute = random.randint(0, 59)
        order_times.append(
            START_DATE + timedelta(days=day, hours=hour-8, minutes=minute)
        )

    is_peak = [1 if t.hour in PEAK_HOURS else 0 for t in order_times]

    # [원본 유지] product_id 생성 방식
    product_ids = [
        f'PRD-{str(random.randint(1, 500)).zfill(4)}'
        for _ in range(n)
    ]

    # [원본 유지] 주문별 category 난수 생성
    raw_categories = random.choices(
        CATEGORIES,
        weights=CATEGORY_WEIGHTS,
        k=n
    )

    # [원본 유지] sku 번호 생성 방식
    sku_numbers = [random.randint(1, 3) for _ in range(n)]
    sku_ids = [
        f'{product_id}-SKU-{sku_no:02d}'
        for product_id, sku_no in zip(product_ids, sku_numbers)
    ]

    return pd.DataFrame({
        'order_id':     [f'ORD-{str(i).zfill(6)}' for i in range(1, n+1)],
        'product_id':   product_ids,
        'category':     raw_categories,
        'sku_id':       sku_ids,
        'order_time':   order_times,
        'is_peak_time': is_peak,
    })

orders_df = generate_orders(N_ORDERS)

# [v7] 토트별 집품 구조. 각 행은 가상 주문의 SKU 1유닛이며 토트에 10~20유닛 배치.
# 실제 WMS 토트-주문 매핑은 없으므로 시간순으로 묶는 임의 토트 배정임.
# 'order_id'는 기존 코드 호환을 위한 유닛 이벤트 고유 ID. 'unit_id'를 별도로 제공.
def allocate_totes_to_units(df, min_units=TOTE_CAPACITY_MIN, max_units=TOTE_CAPACITY_MAX):
    rng=random.Random(4217)  # 주문·취소·회수 난수에 영향 없게 별도 RNG
    n=len(df)
    order_indices=df.sort_values(['order_time','order_id']).index.to_numpy()
    groups=[];remaining=n
    while remaining>max_units:
        unit_count=rng.randint(min_units,min(max_units,remaining-min_units))
        groups.append(unit_count)
        remaining-=unit_count
    if remaining: groups.append(remaining)
    assert sum(groups)==n and all(min_units<=x<=max_units for x in groups)
    tote_values=np.empty(n,dtype=object)
    seq_values=np.zeros(n,dtype=np.int16)
    count_values=np.zeros(n,dtype=np.int16)
    pointer=0
    for i,count in enumerate(groups,1):
        selected=order_indices[pointer:pointer+count]
        tote_values[selected]=f'TOTE-{i:06d}'
        seq_values[selected]=np.arange(1,count+1)
        count_values[selected]=count
        pointer+=count
    df=df.copy()
    df['unit_id']=[f'UNIT-{i:06d}' for i in range(1,n+1)]
    df['tote_id']=tote_values
    df['tote_unit_no']=seq_values
    df['tote_units_total']=count_values
    return df

orders_df = allocate_totes_to_units(orders_df)
print(f'가상 집품 토트 수: {orders_df["tote_id"].nunique():,} / 토트별 상품 유닛 {TOTE_CAPACITY_MIN}~{TOTE_CAPACITY_MAX}개')

# ============================================================
# [수정] product_id별 category 일관성 보정
# - 같은 상품 바코드는 하나의 상품군만 가져야 하므로,
#   원본에서 처음 관측된 category를 해당 product_id의 대표 category로 사용
# - 별도 난수로 category를 다시 생성하지 않음
# ============================================================
product_category_master = (
    orders_df[['product_id', 'category']]
    .drop_duplicates('product_id', keep='first')
    .copy()
)

product_category_map = (
    product_category_master
    .set_index('product_id')['category']
    .to_dict()
)

orders_df['category'] = orders_df['product_id'].map(product_category_map)

# ============================================================
# [수정] 정상 판매가는 product_id 기준으로 고정
# ============================================================
_price_rng = random.Random(4201)

def make_product_price(category):
    low, high = PRICE_RANGE.get(category, (1000, 10000))
    return int(round(_price_rng.uniform(low, high), -2))

product_master = product_category_master.copy()
product_master['product_price'] = (
    product_master['category'].apply(make_product_price)
)

# ============================================================
# [수정] 유통기한은 product_id + sku_id 기준으로 고정
# ============================================================
sku_expiry_master = (
    orders_df[['product_id', 'sku_id', 'category']]
    .drop_duplicates(['product_id', 'sku_id'])
    .copy()
)

_shelf_rng = random.Random(4202)

def make_remaining_shelf_life(category):
    low, high = SHELF_LIFE_RANGE_DAYS.get(category, (7, 30))
    return _shelf_rng.randint(low, high)

sku_expiry_master['remaining_shelf_life_days'] = (
    sku_expiry_master['category'].apply(make_remaining_shelf_life)
)

sku_expiry_master['expiry_date'] = (
    pd.Timestamp(START_DATE).normalize()
    + pd.to_timedelta(
        sku_expiry_master['remaining_shelf_life_days'],
        unit='D'
    )
)

orders_df = orders_df.merge(
    product_master[['product_id', 'product_price']],
    on='product_id',
    how='left'
)

orders_df = orders_df.merge(
    sku_expiry_master[
        ['product_id', 'sku_id', 'remaining_shelf_life_days', 'expiry_date']
    ],
    on=['product_id', 'sku_id'],
    how='left'
)

print(f"전체 주문 수: {len(orders_df):,}")
print(f"상품 바코드 수(product_id): {orders_df['product_id'].nunique():,}")
print(f"유통기한 구분 조합 수(product_id+sku_id): {len(sku_expiry_master):,}")

# 검증
price_check = orders_df.groupby('product_id')['product_price'].nunique()
category_check = orders_df.groupby('product_id')['category'].nunique()

print(f"product_id별 가격 2개 이상인 상품 수: {(price_check > 1).sum():,}")
print(f"product_id별 category 2개 이상인 상품 수: {(category_check > 1).sum():,}")


가상 집품 토트 수: 10,001 / 토트별 상품 유닛 10~20개
전체 주문 수: 150,000
상품 바코드 수(product_id): 500
유통기한 구분 조합 수(product_id+sku_id): 1,500
product_id별 가격 2개 이상인 상품 수: 0
product_id별 category 2개 이상인 상품 수: 0


## **STEP3 - 취소 발생&유형/사유 부여**

In [4]:
# =============================================
# STEP 3: 취소 발생 & cancel_reason 부여
# =============================================

# 셀 단위 시드 고정
random.seed(43)
np.random.seed(43)

def assign_cancel_reason(row):

    # STEP 1에서 정의한 취소 유형 비율을 기준으로 단순취소 / 운영취소를 먼저 결정한다.                  *** 단순취소는 뭔가 싶긴 한데.. 고객취소로 바꿔야 되나 ㅇㅅㅇ..
    # 운영취소일 경우 세부 사유를 부여하고,그중 오집/미집에만 작업자 유형을 부여한다.

    # 1) STEP 1의 취소 유형 비율 기준으로 단순취소 / 운영취소 결정
    cancel_type = random.choices(
        list(CANCEL_TYPE_WEIGHTS.keys()),
        weights=list(CANCEL_TYPE_WEIGHTS.values()),
        k=1
    )[0]

    # 2) 단순취소면 작업자 유형 없음
    if cancel_type == '단순취소':
        reason = '단순취소'
        w_type = '해당없음'

    # 3) 운영취소면 세부 사유 결정
    else:
        reason = random.choices(
            list(CANCEL_REASON_WEIGHTS.keys()),
            weights=list(CANCEL_REASON_WEIGHTS.values()),
            k=1
        )[0]

        # 4) 오집/미집에만 작업자 유형 부여
        if reason in ['오집', '미집']:
            is_exp = random.random() < WORKER_EXPERIENCED_RATIO
            w_type = '숙련자' if is_exp else '비숙련자'
        else:
            w_type = '해당없음'

    return w_type, reason


# 고객취소 + 운영 문제 발생 후보 사건 샘플링
n_cancel  = int(N_ORDERS * CANCEL_RATE)
cancel_df = orders_df.sample(n=n_cancel, random_state=42).copy().reset_index(drop=True)

# 운영 문제 또는 고객취소 발생 시점 (주문 후 1~30분); cancel_time 열명은 호환성 목적                                                      **출고 완료 까지 대략 30분 이내로 가정
cancel_df['cancel_time'] = cancel_df['order_time'].apply(
    lambda t: t + timedelta(minutes=random.randint(1, 30))
)

# 취소 사유 부여
result = cancel_df.apply(assign_cancel_reason, axis=1)
cancel_df['worker_type']   = [r[0] for r in result]
cancel_df['cancel_reason'] = [r[1] for r in result]

# ------------------------------------------------------------
# [추가] 집품 후 고객취소 분리
# - 전체 취소건 및 운영취소 건수는 그대로 유지한다.
# - 기존 단순취소(=집품 전 고객취소) 중 일부만 '고객취소_집품후'로 재분류한다.
# - 규모는 전체 고객취소의 10%으로 설정한다.
# - 이 건은 운영취소 KPI의 분모에는 포함하지 않고 재사용 가능한 재고 공급원으로만 사용한다.
# ------------------------------------------------------------
customer_cancel_count = int(cancel_df['cancel_reason'].eq('단순취소').sum())
n_post_pick_customer_cancel = int(round(
    customer_cancel_count * POST_PICK_CUSTOMER_CANCEL_RATIO
))
simple_indices = cancel_df.index[cancel_df['cancel_reason'] == '단순취소']
post_pick_indices = (
    cancel_df.loc[simple_indices]
    .sample(n=min(n_post_pick_customer_cancel, len(simple_indices)), random_state=4305)
    .index
)
cancel_df.loc[post_pick_indices, 'cancel_reason'] = '고객취소_집품후'
cancel_df.loc[post_pick_indices, 'worker_type'] = '해당없음'

print(f"집품 후 고객취소 분리 건수: {len(post_pick_indices):,}건")

# 재고 위치 & 위치 정보 여부
# TODO: 실제 창고 구역 코드로 교체
cancel_df['stock_location'] = [
    f'{random.choice(["A","B","C","D","E"])}-{random.randint(1,20):02d}'
    for _ in range(len(cancel_df))
]
cancel_df['location_known'] = [
    1 if random.random() < 0.60 else 0
    for _ in range(len(cancel_df))
]

print("[ cancel_reason 분포 ]")
print(cancel_df['cancel_reason'].value_counts(normalize=True).round(3))

# v11: 오류 사건과 물리 재고 공급/대체 수요 행을 분리한다.
# 7,500개 원표본과 고객취소 수, 원래의 오집/미집/파손/분실 표본은 유지.
# 타토트 오집 1유닛은 동일 SKU의 부족 수요 1행 + 별도 공급 1행.
# 과집은 원래 계획 유닛이 정상 처리되고 초과 실물 1개만 공급으로 기록.
import hashlib

def picking_error_u(key, salt):
    return int(hashlib.sha256(f'{key}|{salt}'.encode()).hexdigest()[:12],16)/float(16**12-1)

original_cancel_df = cancel_df.copy(deep=True)
# v13: 인접 토트 연결은 기존 오집 대상 선택과 근접 시간대 설정에만 사용.
# 토트별 독립 난수(초 단위), 기존 기준시각 주변 최대 5분 폭(±2.5분), 실측 아님.
# 같은 시각도 가능하며 토트번호 순으로 시간을 증가시키지 않는다.
TOTE_COMPLETION_SPREAD_MIN = 5.0
tote_times=orders_df.groupby('tote_id')['order_time'].max().sort_index()
tote_ids=tote_times.index.tolist()
pick_batches=[tote_ids[i:i+2] for i in range(0,len(tote_ids),2)]
if len(pick_batches[-1])==1 and len(pick_batches)>1:
    pick_batches[-2].extend(pick_batches.pop())
tote_pick_completed={}; neighbor={}
for batch in pick_batches:
    anchor=max(tote_times.loc[t] for t in batch)
    center=60*(1+29*picking_error_u(batch[0],'pick_completion'))
    low=int(np.ceil(max(60,center-30*TOTE_COMPLETION_SPREAD_MIN)))
    high=int(np.floor(min(1800,center+30*TOTE_COMPLETION_SPREAD_MIN)))
    for j,t in enumerate(batch):
        offset=low+min(high-low,int(picking_error_u(t,'v13_independent_pick_completion')*(high-low+1)))
        tote_pick_completed[t]=anchor+pd.Timedelta(seconds=offset)
        neighbor[t]=batch[(j+1)%len(batch)]


def split_pick_errors(base, overpick_share=OVERPICK_SHARE_OF_WRONG_PICK):
    assert 0 <= overpick_share <= 1
    out=base.copy(deep=True)
    out['error_event_id']=out['order_id']
    out['planned_unit_id']=out['unit_id']
    out['physical_unit_id']=out['unit_id']
    out['original_cancel_reason']=out['cancel_reason']
    out['picking_error_type']='해당없음'
    out['expected_tote_id']=out['tote_id']
    out['actual_tote_id']=out['tote_id']
    # expected/picked는 전체 토트 수량이 아니라 오류 유닛에 대한 수량.
    out['expected_qty']=np.nan
    out['picked_qty']=np.nan
    wrong=out['cancel_reason'].eq('오집')
    missing=out['cancel_reason'].eq('미집')
    out.loc[missing,'picking_error_type']='부족집품'
    out.loc[missing,'expected_qty']=1
    out.loc[missing,'picked_qty']=0
    out.loc[missing,'actual_tote_id']=None
    out.loc[missing,'physical_unit_id']=None
    # 시간순으로 인접한 다른 예정 토트 선택. 물리적 실제 토트 이동 로그는 없음.
    tote_size=orders_df.groupby('tote_id').size().to_dict()
    extra=[]
    for idx,r in out.loc[wrong].iterrows():
        oid=r['order_id']
        if picking_error_u(oid,'overpick_split') < overpick_share:
            out.at[idx,'cancel_reason']='과집'
            out.at[idx,'picking_error_type']='과집'
            out.at[idx,'expected_qty']=0
            out.at[idx,'picked_qty']=1
            out.at[idx,'physical_unit_id']='EXTRA-'+oid
            out.at[idx,'unit_id']='EXTRA-'+oid
        else:
            dest=neighbor[r['tote_id']]
            out.at[idx,'picking_error_type']='타토트오집'
            out.at[idx,'expected_qty']=1
            out.at[idx,'picked_qty']=0
            out.at[idx,'actual_tote_id']=dest
            # 수요행에는 정상 실물이 없음. 연결된 공급행에 원래 물리 유닛을 기록.
            out.at[idx,'physical_unit_id']=None
            supply=out.loc[idx].copy()
            supply['order_id']=oid+'-SUP'
            supply['unit_id']='SUP-'+oid
            supply['physical_unit_id']=r['unit_id']
            supply['cancel_reason']='타토트오집_공급'
            supply['tote_id']=dest
            supply['tote_unit_no']=np.nan # 초과/오류 실물은 계획 집품 순번 없음
            supply['tote_units_total']=tote_size[dest] # 계획 유닛수
            supply['expected_qty']=0
            supply['picked_qty']=1
            # 오집 실물은 실제 토트의 독립적인 집품 완료시각을 따른다.
            # STEP4에서 실제 토트 ID에 따른 이송·취소존 도착/큐를 별도 산정.
            extra.append(supply)
    if extra:
        out=pd.concat([out,pd.DataFrame(extra)],ignore_index=True)
    out['qty_diff']=out['picked_qty']-out['expected_qty']
    out['demand_flag']=out['cancel_reason'].isin(OP_REASONS).astype(int)
    out['pick_supply_only_flag']=out['cancel_reason'].isin(PICK_SUPPLY_ONLY_REASONS).astype(int)
    assert out['order_id'].is_unique
    assert out.loc[out['pick_supply_only_flag'].eq(1),'physical_unit_id'].is_unique
    return out

cancel_df=split_pick_errors(original_cancel_df)
print('원표본 운영 오류 사건:',int(original_cancel_df['cancel_reason'].isin(OP_REASONS).sum()))
print(cancel_df.groupby('cancel_reason').agg(행수=('order_id','size'),대체수요=('demand_flag','sum'),집품공급=('pick_supply_only_flag','sum')).to_string())


집품 후 고객취소 분리 건수: 525건
[ cancel_reason 분포 ]
cancel_reason
단순취소        0.629
오집          0.170
미집          0.073
고객취소_집품후    0.070
토트분실        0.030
리빈파손        0.016
집품파손        0.012
Name: proportion, dtype: float64
원표본 운영 오류 사건: 2254
                 행수  대체수요  집품공급
cancel_reason                  
고객취소_집품후        525     0     0
과집              113     0   113
단순취소           4721     0     0
리빈파손            120   120     0
미집              547   547     0
오집             1163  1163     0
집품파손             88    88     0
타토트오집_공급       1163     0  1163
토트분실            223   223     0


## **STEP4 — 기존 WMS 경로조회(80%) → 실패 시 취소존 FIFO (v14)**

In [5]:
# =============================================
# STEP 4 (v9): 집품 → 라인 → 취소존 → 도착순 FIFO → 검색·취소 판정
# =============================================
# 공통 원칙:
# 1) 모든 시나리오가 동일한 집품·이송·실물 회수 사건을 공유한다.
# 2) 취소존 작업은 cancel_zone_arrived_at 이전에 시작할 수 없다.
# 3) 토트분실은 먼저 기존 WMS 경로 조회로 80% 조치 가능하다고 가정. 실제 회수가 기한 내 완료되면 취소존 미경유.
#    경로 조회 실패 또는 회수 지연으로 원주문 완료에 실패한 경우에만 라인 도착 5분 뒤 취소존에 입고.
# 4) 원토트 확인 전 관리자 마감(피크 20/일반 30분)이 도래하면,
#    해당 유닛은 다른 동일 SKU 정상 재고 대체 대상으로 넘어간다.
# 5) 수동 탐색 누락 10%는 시간 부족과 별도로 '실물 탐색 실수'로만 사용한다.
# 6) 미회수 9%는 취소 행정처리 후 당일 라인 도착; 1%는 수일 뒤 회수.
import hashlib
import heapq


def stable_uniform_local(key, salt):
    raw = f'{key}|{salt}'.encode('utf-8')
    return int(hashlib.sha256(raw).hexdigest()[:12],16) / float(16**12 - 1)


def tote_recovery_class(order_id):
    u=stable_uniform_local(order_id,'tote_return_within_30')
    if u < TOTE_RETURN_WITHIN_30_PROB: return 'within_30'
    if u < 1 - TOTE_MULTI_DAY_RECOVERY_PROB: return 'late_same_day'
    return 'multi_day'


def manual_attempt_prob_from_delay(delay_min):
    # 타 정상 재고 대체: 작업 부담이 큰 경우 재사용 시도 포기 확률만 적용.
    if delay_min <= DELAY_THRESHOLD_MIN: return 1.0
    if delay_min <= 70: return MANUAL_DELAYED_REUSE_ATTEMPT_PROB_30_TO_70
    return MANUAL_DELAYED_REUSE_ATTEMPT_PROB_OVER_70


def stable_minutes(key, salt, low, high):
    return low+(high-low)*stable_uniform_local(key,salt)


def prepare_causal_events(base_df, process_type='manual'):
    out=base_df.copy().reset_index(drop=True)
    # STEP3의 원래 cancel_time은 '주문 후 1~30분'의 가상 집품완료 시점으로 해석.
    out['pick_completed_at']=out['tote_id'].map(tote_pick_completed)
    out['line_transit_min']=out['tote_id'].map(
        lambda k:stable_minutes(k,'line_transit',*LINE_TRANSIT_MIN))
    out['expected_line_arrived_at']=out['pick_completed_at']+pd.to_timedelta(out['line_transit_min'],unit='m')
    out['problem_detected_at']=out['expected_line_arrived_at']
    # 기존 cancel_time 열은 v9부터 운영 문제 '인지 시각' 의미로만 사용.
    out['cancel_time']=out['problem_detected_at']
    out['cancel_zone_transit_min']=out['tote_id'].map(
        lambda k:stable_minutes(k,'zone_transit',*CANCEL_ZONE_TRANSIT_MIN))
    out['line_arrived_at']=out['expected_line_arrived_at']
    out['cancel_zone_arrived_at']=out['line_arrived_at']+pd.to_timedelta(out['cancel_zone_transit_min'],unit='m')

    # 무실물 부족 수요: 일반 이송 구간을 요청 전달시간의 대용으로 사용.
    absent=out['cancel_reason'].isin(['오집','미집'])
    out['demand_request_at']=pd.NaT
    out.loc[absent,'demand_request_at']=out.loc[absent,'cancel_zone_arrived_at']
    out.loc[absent,'cancel_zone_arrived_at']=pd.NaT
    out['physical_fifo_service_count']=0
    out['physical_fifo_service_min']=0.0
    out['virtual_fifo_service_count']=0
    is_lost=out['cancel_reason'].eq('토트분실')
    out['tote_recovery_class']='not_applicable'
    out['tote_return_within_30_flag']=np.nan
    out['tote_delayed_return_flag']=0
    out['tote_multi_day_recovery_flag']=0
    out['tote_return_time']=pd.NaT
    out['tote_cancel_deadline']=pd.NaT
    out['tote_search_detect_time']=pd.NaT
    out['tote_predecision_wait_min']=np.nan
    out['tote_predecision_workload_delay_min']=np.nan
    out['tote_manual_attempt_prob']=np.nan
    out['tote_manual_search_found_flag']=np.nan
    out['tote_rescue_tool_used_flag']=0
    out['tote_existing_route_traced_flag']=0  # 기존 WMS 경로 조회로 위치를 파악할 수 있는 토트
    out['tote_existing_route_rescued_flag']=0 # 기존 기능으로 제때 원주문 완료, 새 도구 성과 아님
    out['tote_cancel_zone_required_flag']=0   # 원래 라인 조치 실패 시에만 취소존 이송
    out['tote_rescue_path']='none'
    out['tote_rescue_flag']=0
    out['cancel_zone_processed_at']=pd.NaT
    out['cancel_zone_queue_started_at']=pd.NaT
    out['cancel_zone_queue_wait_min']=np.nan
    out['queue_entry_time']=out['cancel_zone_arrived_at']
    out['process_time']=pd.NaT
    out['process_delay_min']=np.nan
    out['queue_processing_started_at']=pd.NaT
    out['process_type']=process_type
    out['tote_manual_search_found_flag']=np.nan

    for idx,row in out.loc[is_lost].iterrows():
        oid=row['order_id']
        kind=tote_recovery_class(oid)
        out.at[idx,'tote_recovery_class']=kind
        out.at[idx,'tote_existing_route_traced_flag']=int(stable_uniform_local(oid,'legacy_tote_route_trace')<EXISTING_TOTE_ROUTE_TRACE_PROB)
        out.at[idx,'tote_cancel_zone_required_flag']=1  # 성공시 사전 판정에서 0으로 변경
        out.at[idx,'tote_return_within_30_flag']=int(kind=='within_30')
        out.at[idx,'tote_delayed_return_flag']=int(kind=='late_same_day')
        out.at[idx,'tote_multi_day_recovery_flag']=int(kind=='multi_day')
        detected=pd.Timestamp(row['problem_detected_at'])
        deadline=detected+pd.Timedelta(minutes=TOTE_WAIT_PEAK_MIN if detected.hour in PEAK_HOURS else TOTE_WAIT_NORMAL_MIN)
        out.at[idx,'tote_cancel_deadline']=deadline
        if kind=='within_30':
            # 수동/자동에 동일한 물리 회수 시간 고정: 분실 인지 후 +1~30분 라인 입고.
            returned=detected+pd.Timedelta(minutes=stable_minutes(oid,'tote_return_time',1.0,30.0))
            out.at[idx,'tote_return_time']=returned
            out.at[idx,'line_arrived_at']=returned
            out.at[idx,'cancel_zone_arrived_at']=returned+pd.Timedelta(minutes=LOST_TOTE_TO_ZONE_MIN)
        else:
            # 마감 이후 행정 취소/대체 작업 종료 후 회수시간을 결정하므로 아직 미정.
            out.at[idx,'line_arrived_at']=pd.NaT
            out.at[idx,'cancel_zone_arrived_at']=pd.NaT
        out.at[idx,'queue_entry_time']=deadline  # 원주문 복구 실패 시에만 행정 처리 큐 유효
    out['source_assignment_released_flag']=0
    return out


def decide_restock(row):
    reason=row['cancel_reason']
    if reason not in SUPPLY_REASONS: return 0
    if int(row.get('tote_rescue_flag',0))==1:return 0
    if reason=='고객취소_집품후':return 1
    physical_wait=row.get('cancel_zone_queue_wait_min',np.nan)
    delay=(float(physical_wait)+float(row.get('physical_fifo_service_min',0))) if pd.notna(physical_wait) else float(row['process_delay_min'])
    miss_prob=min(0.05+(delay/100)*0.10,0.10)
    return int(stable_uniform_local(row['order_id'],'restock')>=miss_prob)


def build_scenario_pre_match(base_df, process_type='manual',tool_usage_rate=1.0,include_virtual_work=False):
    """실물만 처리 용량 소비. 부족/마감 요청은 선행 실물 처리 대기만 참조(0분 서비스).
    include_virtual_work=True는 동일 표본에서 중복 부하를 비교하는 진단 전용."""
    assert process_type in ('manual','automated') and 0<=tool_usage_rate<=1
    out=prepare_causal_events(base_df,process_type)
    # 신규 검색툴과 무관하게 기존 WMS 토트 경로 확인으로 원주문을 회복한 사건을
    # 양 시나리오에 똑같이 선행 판정한다. 도착/확인 완료가 관리자 마감 전이어야 한다.
    early = out['cancel_reason'].eq('토트분실') & out['tote_recovery_class'].eq('within_30')
    route_valid = early & out['tote_existing_route_traced_flag'].eq(1)
    route_detect_time = out['tote_return_time'] + pd.Timedelta(minutes=EXISTING_ROUTE_VERIFY_MIN)
    quality_ok=out['order_id'].map(lambda oid:stable_uniform_local(oid,'inspection')<0.90)
    route_rescued = route_valid & route_detect_time.le(out['tote_cancel_deadline']) & quality_ok
    out.loc[route_rescued,'tote_rescue_flag']=1
    out.loc[route_rescued,'tote_existing_route_rescued_flag']=1
    out.loc[route_rescued,'tote_rescue_path']='기존 WMS 경로조회'
    out.loc[route_rescued,'tote_cancel_zone_required_flag']=0
    out.loc[route_rescued,'tote_search_detect_time']=route_detect_time.loc[route_rescued]
    out.loc[route_rescued,'tote_predecision_wait_min']=0.0
    out.loc[route_rescued,'tote_predecision_workload_delay_min']=0.0
    out.loc[route_rescued,'tote_manual_attempt_prob']=1.0
    # 원래 주문으로 복귀했으므로 취소존에는 가지 않고 행정 취소/공급원에서도 제외한다.
    out.loc[route_rescued,'cancel_zone_arrived_at']=pd.NaT
    out.loc[route_rescued,'queue_entry_time']=pd.NaT
    if process_type=='automated':
        out['location_known']=out['order_id'].map(
            lambda oid:int(stable_uniform_local(oid,'auto_location_known')<AUTOMATED_LOCATION_KNOWN_PROB))
    time_per_case=MANUAL_TIME_PER_CASE if process_type=='manual' else AUTOMATED_TIME_PER_CASE
    rng=random.Random(44 if process_type=='manual' else 52)
    batch_limit=rng.randint(30,50)
    batch_count=0
    available=pd.Timestamp.min+pd.Timedelta(days=10)
    heap=[]; sequence=0
    def enqueue(ts,idx,job):
        nonlocal sequence
        if pd.isna(ts):return
        sequence+=1
        heapq.heappush(heap,(pd.Timestamp(ts),sequence,idx,job))

    for idx,row in out.iterrows():
        reason=row['cancel_reason']
        if reason in ('오집','미집'):
            enqueue(row['demand_request_at'],idx,'demand_request')
        elif reason in (['집품파손','리빈파손','고객취소_집품후'] + PICK_SUPPLY_ONLY_REASONS):
            enqueue(row['cancel_zone_arrived_at'],idx,'normal_error')
        elif reason=='토트분실' and row['tote_cancel_zone_required_flag']==1:
            out.at[idx,'demand_request_at']=row['tote_cancel_deadline']
            enqueue(row['tote_cancel_deadline'],idx,'lost_deadline')
            if row['tote_recovery_class']=='within_30':
                enqueue(row['cancel_zone_arrived_at'],idx,'lost_physical')
        # 집품 후 고객취소는 실제 실물 FIFO 1회; 집품 전은 서비스 없음.

    while heap:
        entry,_,idx,job=heapq.heappop(heap)
        oid=out.at[idx,'order_id']
        if job=='lost_deadline' and out.at[idx,'tote_rescue_flag']==1:
            continue # 마감 전에 원래 상품 확보한 경우 행정취소 작업 발생하지 않음.
        physical=job in ('normal_error','lost_physical','late_physical')
        consumes=physical or include_virtual_work
        started=max(entry,available)
        completed=started+pd.Timedelta(minutes=time_per_case if consumes else 0)
        if consumes:
            available=completed
            if physical:
                out.at[idx,'physical_fifo_service_count']+=1
                out.at[idx,'physical_fifo_service_min']+=time_per_case
            else:
                out.at[idx,'virtual_fifo_service_count']+=1
            batch_count+=1
            if process_type=='manual' and batch_count>=batch_limit:
                available+=pd.Timedelta(minutes=rng.uniform(2.0,5.0))
                batch_count=0
                batch_limit=rng.randint(30,50)
        if job in ('normal_error','lost_physical','late_physical'):
            out.at[idx,'cancel_zone_queue_started_at']=started
        if job=='normal_error':
            out.at[idx,'queue_entry_time']=entry
            out.at[idx,'queue_processing_started_at']=started
            out.at[idx,'process_time']=completed
            out.at[idx,'process_delay_min']=(completed-entry).total_seconds()/60
            out.at[idx,'cancel_zone_processed_at']=completed
            out.at[idx,'cancel_zone_queue_wait_min']=(started-entry).total_seconds()/60
        elif job=='demand_request':
            out.at[idx,'queue_entry_time']=entry
            out.at[idx,'queue_processing_started_at']=started
            out.at[idx,'process_time']=completed
            out.at[idx,'process_delay_min']=(completed-entry).total_seconds()/60
        elif job=='lost_physical' or job=='late_physical':
            # 원래 토트가 취소존에 도착한 실제 시각부터만 대기하고 작업한다.
            out.at[idx,'cancel_zone_processed_at']=completed
            out.at[idx,'cancel_zone_queue_wait_min']=(started-entry).total_seconds()/60
            if job=='lost_physical':
                used_tool=(process_type=='automated' and
                           stable_uniform_local(oid,'tool_adoption')<tool_usage_rate)
                out.at[idx,'tote_rescue_tool_used_flag']=int(used_tool)
                direct_search=bool(used_tool and out.at[idx,'location_known']==1)
                verify=AUTOMATED_TOTE_DETECTION_MIN if direct_search else stable_minutes(
                    oid,'onsite_verify',*MANUAL_ONSITE_VERIFY_RANGE)
                detected=completed+pd.Timedelta(minutes=verify)
                out.at[idx,'tote_search_detect_time']=detected
                out.at[idx,'tote_predecision_wait_min']=(started-out.at[idx,'problem_detected_at']).total_seconds()/60
                out.at[idx,'tote_predecision_workload_delay_min']=(started-entry).total_seconds()/60
                out.at[idx,'tote_manual_attempt_prob']=1.0  # 마감초과 판단과 별도 포기확률 중복금지
                found=int(direct_search or stable_uniform_local(oid,'manual_match_search')>=MANUAL_MATCH_MISS_PROB)
                out.at[idx,'tote_manual_search_found_flag']=found
                physical_normal=stable_uniform_local(oid,'inspection')<0.90
                if detected<=out.at[idx,'tote_cancel_deadline'] and physical_normal and found:
                    out.at[idx,'tote_rescue_flag']=1
                    out.at[idx,'tote_rescue_path']='취소존 수동확인' if not direct_search else '신규 검색툴 확인'
                    out.at[idx,'tote_cancel_zone_required_flag']=1  # 실제 취소존으로 이동했고 현장에서 복구함
                    out.at[idx,'queue_entry_time']=pd.NaT
                    out.at[idx,'process_time']=pd.NaT
                    out.at[idx,'process_delay_min']=np.nan
            # late_physical은 관리자 마감 이후이므로 원 주문 복구 기여 불가.
        elif job=='lost_deadline':
            out.at[idx,'queue_entry_time']=entry
            out.at[idx,'queue_processing_started_at']=started
            out.at[idx,'process_time']=completed
            out.at[idx,'process_delay_min']=(completed-entry).total_seconds()/60
            kind=out.at[idx,'tote_recovery_class']
            if kind=='late_same_day':
                later=completed+pd.Timedelta(minutes=stable_minutes(
                    oid,'forced_cancel_return_lag',*TOTE_LATE_SAME_DAY_RETURN_DELAY_RANGE))
                out.at[idx,'tote_return_time']=later
                out.at[idx,'line_arrived_at']=later
                zone=later+pd.Timedelta(minutes=LOST_TOTE_TO_ZONE_MIN)
                out.at[idx,'cancel_zone_arrived_at']=zone
                enqueue(zone,idx,'late_physical')
            elif kind=='multi_day':
                u=stable_uniform_local(oid,'multi_day_recovery_lag')
                lo,hi=TOTE_MULTI_DAY_RECOVERY_LAG_DAYS
                days=lo+int(u*(hi-lo+1))
                later=completed+pd.Timedelta(days=days)
                out.at[idx,'tote_return_time']=later
                out.at[idx,'line_arrived_at']=later
                out.at[idx,'cancel_zone_arrived_at']=later+pd.Timedelta(minutes=LOST_TOTE_TO_ZONE_MIN)
                # 수일 뒤 도착은 기준 단일 운영일의 큐/공급원에 편입하지 않음.
    # 집품 전 고객취소는 실물 없음. 집품 후는 위 normal_error에서 이미 처리.
    customer=out['cancel_reason'].eq('단순취소')
    out.loc[customer,['cancel_zone_arrived_at','queue_entry_time','process_time']]=pd.NaT
    out.loc[customer,'process_delay_min']=0.0
    out['customer_cancel_return_time']=pd.NaT
    customer_post=out['cancel_reason'].eq('고객취소_집품후')
    out.loc[customer_post,'customer_cancel_return_time']=out.loc[customer_post,'cancel_zone_arrived_at']
    # 원토트 복구된 건은 재사용 공급이 아니며; 나머지는 행정 처리 후 할당 해제 가능.
    out['source_assignment_released_flag']=(
        out['cancel_reason'].isin(SUPPLY_REASONS) & out['tote_rescue_flag'].eq(0)).astype(int)
    out['restock_flag']=out.apply(decide_restock,axis=1)
    out=out.sort_values(['cancel_time','order_id']).reset_index(drop=True)
    return out

# 수동 · 자동 공통 물리 사건 및 시드를 공유. STEP 3의 원본 표본은 보존한다.
base_cancel_df=cancel_df.copy(deep=True)
cancel_df=build_scenario_pre_match(base_cancel_df,'manual',1.0)
print('STEP 4 취소 전 원토트 복구 수동:',int(cancel_df['tote_rescue_flag'].sum()))
print('STEP 4 실물 FIFO 서비스:',int(cancel_df['physical_fifo_service_count'].sum()))
print('STEP 4 취소존 선행 도착 기반 처리:',int(cancel_df['cancel_zone_processed_at'].notna().sum()),'건')


STEP 4 취소 전 원토트 복구 수동: 138
STEP 4 실물 FIFO 서비스: 2104
STEP 4 취소존 선행 도착 기반 처리: 2104 건


## **STEP5 - 검수/재사용/추가 취소**

In [6]:
# =============================================
# STEP 5. 동일 사건 비교 · 정상 실물 재고 1:1 매칭
# =============================================
# 사건 단위 = 단일 SKU의 운영취소 이벤트 (다품목 주문 전체 취소 아님)
# 운영 문제 발생(=cancel_time)과 취소 결정(=queue_entry_time)을 구분.
# 원 토트 복구는 별도 플래그, 취소 확정 후 할당해제 상품만 타 주문 공급원.
# 토트 자체가 아닌 회수된 SKU 1개를 공급 단위로 관리한다.
import hashlib
INSPECTION_PASS_RATE = 0.90

def deterministic_uniform(key, salt=''):
    raw = f'{key}|{salt}'.encode('utf-8')
    return int(hashlib.sha256(raw).hexdigest()[:12], 16) / float(16**12 - 1)

def operating_day(ts):
    """운영일: 08:00~익일 02:00. 02:00~07:59는 유효 운영일 없음."""
    ts = pd.Timestamp(ts)
    if pd.isna(ts):
        return pd.NaT
    if ts.hour < 2:
        return (ts - pd.Timedelta(days=1)).normalize()
    if ts.hour >= 8:
        return ts.normalize()
    return pd.NaT


def assign_physical_inspection_flag(df):
    """등록 누락과 독립된 실물 품질. 같은 order_id면 양 시나리오에서 동일."""
    return [int((r['cancel_reason'] in SUPPLY_REASONS)
                and (r['cancel_reason'] != '토트분실'
                     or pd.notna(r['tote_return_time']))
                and deterministic_uniform(r['order_id'], 'inspection') < INSPECTION_PASS_RATE)
            for _, r in df.iterrows()]


def assign_inspection_flag(df):
    """재고 등록·검수 통과를 모두 충족한 전산 검색 공급 후보."""
    return (df['physical_inspection_pass_flag'].astype(int)
            * df['restock_flag'].astype(int)).tolist()


cancel_df['physical_inspection_pass_flag'] = assign_physical_inspection_flag(cancel_df)
cancel_df['inspection_flag'] = assign_inspection_flag(cancel_df)

# 할당 해제 여부는 STEP 4에서 원 토트 복구 판단 이후 설정함.
assert not ((cancel_df['tote_rescue_flag']==1)&(cancel_df['source_assignment_released_flag']==1)).any()

def get_manual_reuse_attempt_prob(process_delay_min):
    if pd.isna(process_delay_min): return 0.0
    return manual_attempt_prob_from_delay(float(process_delay_min))

def assign_reuse_with_time_window(df, process_type='manual', tool_usage_rate=1.0):
    """취소존 FIFO 처리완료 후 같은 운영일 기존재고 우선, 이후 30분 관찰창.
    실제 가용시간 이전 사용은 금지하며, 공급원은 한번만 소비한다.
    도구 미사용(부분 활용 실험) 시 자동 처리 큐는 유지하고 현장 수동 검색을 수행한다.
    """
    assert 0 <= tool_usage_rate <= 1
    result = df.copy().reset_index(drop=True)
    result['_row_key'] = np.arange(len(result))
    source_mask = (result['cancel_reason'].isin(SUPPLY_REASONS)
                   & (result['source_assignment_released_flag'] == 1)
                   & (result['restock_flag'] == 1)
                   & (result['inspection_flag'] == 1))
    sources = result.loc[source_mask, ['_row_key', 'order_id', 'product_id', 'sku_id',
        'cancel_reason','cancel_time','process_time','tote_return_time',
        'customer_cancel_return_time','queue_entry_time','cancel_zone_processed_at']].copy()
    sources['available_time'] = sources['process_time']
    # 물리적으로 취소존에 도착하여 검수·등록 처리가 끝나기 전에는 공급 불가.
    sources.loc[:, 'available_time'] = sources[['process_time','cancel_zone_processed_at']].max(axis=1)
    tmask = sources['cancel_reason'].eq('토트분실')
    # 취소 전에는 원래 주문에 할당된 토트. 취소·전산처리 완료 전에 타 주문 공급 불가.
    sources.loc[tmask, 'available_time'] = sources.loc[tmask,['tote_return_time','process_time','cancel_zone_processed_at','queue_entry_time']].max(axis=1)
    # 집품 후 고객취소 역시 FIFO 처리 완료 후 공급 가능.
    sources['operating_day'] = sources['available_time'].map(operating_day)
    sources = sources[sources['operating_day'].notna()].copy()
    sources_by_sku = {}
    for source in sources.sort_values(['available_time','order_id']).to_dict('records'):
        sources_by_sku.setdefault((source['product_id'],source['sku_id']),[]).append(source)

    used_source_ids = set()
    maps = {k:{} for k in ['reused_flag','time_window_match_flag','matched_candidate_order_id',
        'consumed_candidate_order_id','matched_candidate_available_time','match_time_gap_min',
        'manual_search_success_flag','delay_reuse_attempt_prob','final_reuse_prob_if_matched',
        'tool_used_flag','reuse_ready_time','inventory_wait_after_processing_min']}
    targets = result[(result['cancel_reason'].isin(OP_REASONS))
        & (result['tote_rescue_flag'] == 0)].sort_values(['process_time','queue_entry_time','order_id'])
    for _, row in targets.iterrows():
        key = row['_row_key']
        tool_used = int(process_type == 'automated' and
            deterministic_uniform(row['order_id'], 'tool_adoption') < tool_usage_rate)
        maps['tool_used_flag'][key] = tool_used
        # 재고를 실제 조회할 수 있는 것은 취소존 큐 처리 이후다.
        # 큐 진입 전 시점에서 미래 30분을 관찰하던 과거 모델은 수정한다.
        target_event_time = pd.Timestamp(row['process_time'])
        day = operating_day(target_event_time)
        if pd.isna(day): continue
        group = sources_by_sku.get((row['product_id'],row['sku_id']))
        if group is None: continue
        # 정렬된 후보를 한 번만 훑는다. 기존 FIFO/미래 30분 선택 규칙과 동일.
        cand = None
        cutoff = target_event_time + pd.Timedelta(minutes=MATCH_WINDOW_MIN)
        for candidate in group:
            if candidate['available_time'] > cutoff:
                break
            if (candidate['_row_key'] not in used_source_ids and
                candidate['order_id'] != row['order_id'] and
                candidate['operating_day'] == day):
                cand = candidate
                break
        if cand is None:
            continue
        maps['time_window_match_flag'][key] = 1
        maps['matched_candidate_order_id'][key] = cand['order_id']  # 후보 확인(소비 전)
        maps['matched_candidate_available_time'][key] = cand['available_time']
        maps['match_time_gap_min'][key] = (cand['available_time'] - target_event_time).total_seconds()/60

        # 전산 검색을 사용하지 않은 대상은 동일한 운영 상황에서 현장 수동탐색으로 전환.
        # 자동 처리량/재고 반영률은 유지; '검색 수행률'만 분리한 민감도 실험임.
        use_manual_search = process_type == 'manual' or not tool_used or (row['location_known'] != 1)
        if use_manual_search:
            found = int(deterministic_uniform(row['order_id'], 'manual_match_search')
                        >= MANUAL_MATCH_MISS_PROB)
            maps['manual_search_success_flag'][key] = found
            if not found:
                maps['final_reuse_prob_if_matched'][key] = 0.0
                continue
            attempt_prob = get_manual_reuse_attempt_prob(row['process_delay_min'])
            maps['delay_reuse_attempt_prob'][key] = attempt_prob
            maps['final_reuse_prob_if_matched'][key] = (1-MANUAL_MATCH_MISS_PROB)*attempt_prob
            if deterministic_uniform(row['order_id'], 'manual_delay_reuse_attempt') >= attempt_prob:
                continue
        else:
            maps['manual_search_success_flag'][key] = 1
            maps['delay_reuse_attempt_prob'][key] = 1.0
            maps['final_reuse_prob_if_matched'][key] = 1.0
        maps['reused_flag'][key] = 1
        maps['consumed_candidate_order_id'][key] = cand['order_id']
        ready = max(pd.Timestamp(row['process_time']), pd.Timestamp(cand['available_time']))
        maps['reuse_ready_time'][key] = ready
        maps['inventory_wait_after_processing_min'][key] = max(0,
            (pd.Timestamp(cand['available_time']) - pd.Timestamp(row['process_time'])).total_seconds()/60)
        used_source_ids.add(int(cand['_row_key']))

    fill_zero = ['reused_flag','time_window_match_flag','tool_used_flag']
    for col, value_map in maps.items():
        result[col] = result['_row_key'].map(value_map)
        if col in fill_zero:
            result[col] = result[col].fillna(0).astype(int)
    # 토트 복구는 타 재고 대체 매칭 루프를 거치지 않으나 도구 활용률 집계에는 포함된다.
    rescued=result['tote_rescue_flag'].eq(1)
    result.loc[rescued,'tool_used_flag']=result.loc[rescued,'tote_rescue_tool_used_flag'].astype(int)
    return result.drop(columns=['_row_key'])

def add_physical_flags(df):
    result=df.copy()
    result['physical_inspection_pass_flag']=assign_physical_inspection_flag(result)
    result['inspection_flag']=assign_inspection_flag(result)
    return result

cancel_df = add_physical_flags(cancel_df)
cancel_df = assign_reuse_with_time_window(cancel_df, 'manual')

def decide_add_cancel(row):
    # 추가취소는 고객의 최종 구매취소가 아니라 공정의 취소/재할당 재인입 이벤트
    if row['cancel_reason'] not in OP_REASONS: return 0
    # 물리적으로 자기 토트가 늦게 와도 기존 정상 재고로 대체하면 원래 유닛의 취소를 예방할 수 있다.
    return 1 - int(row['reused_flag']) - int(row['tote_rescue_flag'])

def add_operational_outcomes(df):
    out=df.copy()
    out['additional_cancel_flag']=out.apply(decide_add_cancel,axis=1)
    out['op_cancel_prevented_flag']=(out['reused_flag'].fillna(0).astype(int)+out['tote_rescue_flag'].astype(int)).astype(int)
    out['op_cancel_confirmed_flag']=out['additional_cancel_flag'].astype(int)
    assert (out['op_cancel_prevented_flag']<=1).all()
    assert (out.loc[out['cancel_reason'].isin(OP_REASONS),'op_cancel_prevented_flag']+out.loc[out['cancel_reason'].isin(OP_REASONS),'op_cancel_confirmed_flag']==1).all()
    return out

cancel_df = add_operational_outcomes(cancel_df)
print('STEP 5 수동 타 주문 재고 대체 건수:', int(cancel_df.loc[cancel_df['cancel_reason'].isin(OP_REASONS), 'reused_flag'].sum()))
print('STEP 5 수동 원 토트 복구 건수:',int(cancel_df['tote_rescue_flag'].sum()))
print('STEP 5 수동 운영취소 확정 건수:',int(cancel_df['op_cancel_confirmed_flag'].sum()))
print('STEP 5 상품 회수 검수와 시스템 반영 검수의 차이:', int((cancel_df['physical_inspection_pass_flag'] != cancel_df['inspection_flag']).sum()))


STEP 5 수동 타 주문 재고 대체 건수: 1026
STEP 5 수동 원 토트 복구 건수: 138
STEP 5 수동 운영취소 확정 건수: 977
STEP 5 상품 회수 검수와 시스템 반영 검수의 차이: 210


## **STEP6 — 실제 취소존 대기시간과 수동 탐색 소요시간 분리**

In [7]:
# =============================================
# STEP 6. 탐색 효율 KPI (전체 운영취소 대상, 성공/실패 무관)
# =============================================
# search_time_min = 순수 검색 + 현장확인(수동인 경우).
# 취소존의 FIFO 대기시간은 process_delay_min과 cancel_zone_queue_wait_min에 따로 존재하며
# 검색시간에 임의 10~60분 대기를 중복해서 얹지 않는다.
# 미래에 입고되는 실물 대기는 inventory_wait_after_processing_min 별도 지표.
def calc_search(row):
    if row['cancel_reason'] not in OP_REASONS or row['tote_rescue_flag']==1:
        return None,None,None,None
    search_start=pd.Timestamp(row['process_time'])
    u=deterministic_uniform(row['order_id'],'search_duration')
    basic=round(u if bool(row['location_known']) else 1+2*u,2)
    tool_used=(row.get('process_type','manual')=='automated' and
        row.get('tool_used_flag',0)==1 and row['location_known']==1)
    verify=0.0 if tool_used else stable_minutes(row['order_id'],'onsite_verify',*MANUAL_ONSITE_VERIFY_RANGE)
    total=round(basic+verify,2)
    # 취소존 도착 이후 큐 대기는 process_delay_min에 별도로 포함; 탐색시간과 중복 가산하지 않음.
    return search_start,search_start+pd.Timedelta(minutes=total),total,0.0

s = cancel_df.apply(calc_search, axis=1)
cancel_df['search_start_time'] = [r[0] for r in s]
cancel_df['search_end_time'] = [r[1] for r in s]
cancel_df['search_time_min'] = [r[2] for r in s]
cancel_df['search_wait_min'] = [r[3] for r in s]
COLS = [
    'order_id','unit_id','error_event_id','planned_unit_id','physical_unit_id','original_cancel_reason','picking_error_type','expected_tote_id','actual_tote_id','expected_qty','picked_qty','qty_diff','demand_flag','pick_supply_only_flag','tote_id','tote_unit_no','tote_units_total','product_id','category','sku_id','product_price','expiry_date','remaining_shelf_life_days',
    'worker_type','is_peak_time','cancel_reason','cancel_time','pick_completed_at','expected_line_arrived_at','problem_detected_at',
    'line_arrived_at','cancel_zone_arrived_at','cancel_zone_queue_started_at','cancel_zone_processed_at',
    'cancel_zone_queue_wait_min','queue_processing_started_at','demand_request_at','physical_fifo_service_count','physical_fifo_service_min','virtual_fifo_service_count',
    'stock_location','location_known',
    'process_type','process_time','process_delay_min',
    'restock_flag','physical_inspection_pass_flag','inspection_flag','source_assignment_released_flag',
    'queue_entry_time','tote_cancel_deadline','tote_search_detect_time','tote_predecision_wait_min','tote_predecision_workload_delay_min','tote_manual_attempt_prob','tote_manual_search_found_flag','tote_rescue_tool_used_flag','tote_rescue_flag',
    'tote_existing_route_traced_flag','tote_existing_route_rescued_flag','tote_cancel_zone_required_flag','tote_rescue_path',
    'tote_return_within_30_flag','tote_recovery_class','tote_delayed_return_flag','tote_multi_day_recovery_flag',
    'tote_return_time','customer_cancel_return_time',
    'search_start_time','search_end_time','search_wait_min','search_time_min',
    'reused_flag','additional_cancel_flag','op_cancel_prevented_flag','op_cancel_confirmed_flag','time_window_match_flag','matched_candidate_order_id',
    'consumed_candidate_order_id','matched_candidate_available_time','match_time_gap_min',
    'manual_search_success_flag','delay_reuse_attempt_prob','final_reuse_prob_if_matched',
    'tool_used_flag','reuse_ready_time','inventory_wait_after_processing_min'
]
final_df = cancel_df[COLS].copy()
print('STEP 6 수동 데이터:', final_df.shape)
print('취소처리 대상 탐색 시간 기록 완전성:',final_df.loc[final_df['cancel_reason'].isin(OP_REASONS)&final_df['tote_rescue_flag'].eq(0),'search_time_min'].notna().mean())
print('※ 원 토트 복구건은 취소처리 이후 탐색이 없으며, 별도 선행 탐색시각을 기록함')


STEP 6 수동 데이터: (8663, 87)
취소처리 대상 탐색 시간 기록 완전성: 1.0
※ 원 토트 복구건은 취소처리 이후 탐색이 없으며, 별도 선행 탐색시각을 기록함


## **STEP7 - 1단계 분석**

In [8]:
# =============================================
# STEP 7: 1단계 분석 — 전체 현황
# =============================================
import matplotlib.gridspec as gridspec

op_df = final_df[final_df['demand_flag'].eq(1)].copy()  # 대체/복구 수요만

print("=" * 45)
print("1단계. 전체 현황")
print("=" * 45)
print(f"\n고객취소+오류 수요+공급 처리행 합계: {len(final_df):,}건")
print(f"운영 문제 후보 건수: {len(op_df):,}건")
print(f"원 토트 복구 건수: {int(op_df['tote_rescue_flag'].sum()):,}건")
print(f"운영취소 확정 건수: {int(op_df['op_cancel_confirmed_flag'].sum()):,}건")
print(f"운영취소 예방률: {op_df['op_cancel_prevented_flag'].mean():.1%}")
print(f"타 정상 SKU 대체율: {op_df['reused_flag'].mean():.1%}")
print(f"운영취소 확정률: {op_df['op_cancel_confirmed_flag'].mean():.1%}")

print("\n[ 취소 사유별 비중 ]")
print(final_df['cancel_reason'].value_counts(normalize=True).round(3))

# =========================
# 그래프 생성
# =========================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# =========================
# 왼쪽: 취소 사유 (로그 스케일 적용)
# =========================
counts = final_df['cancel_reason'].value_counts()

bars = axes[0].bar(
    counts.index,
    counts.values,
    color=['#4C72B0','#DD8452','#55A868','#C44E52','#9467BD'][:len(counts)]
)

axes[0].set_title('취소 사유별 건수')
axes[0].set_yscale('log')  # 핵심 (엑셀 느낌)
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_ylabel('건수 (log scale)')

# 값 라벨 표시
for bar in bars:
    height = bar.get_height()
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        height,
        f'{int(height):,}',
        ha='center',
        va='bottom',
        fontsize=9
    )

# =========================
# 오른쪽: 운영 취소 핵심 지표
# =========================
reuse_rate    = op_df['reused_flag'].mean()
add_can_rate  = op_df['additional_cancel_flag'].mean()

axes[1].bar(
    ['타 재고 대체', '원 토트 복구', '운영취소 확정'],
    [reuse_rate, op_df['tote_rescue_flag'].mean(), add_can_rate],
    color=['#55A868','#C44E52','#DD8452']
)

axes[1].set_title('운영 문제 후보 중 처리 결과 (합계 100%)')
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f'{y:.0%}')
)

plt.tight_layout()
plt.show()

1단계. 전체 현황

고객취소+오류 수요+공급 처리행 합계: 8,663건
운영 문제 후보 건수: 2,141건
원 토트 복구 건수: 138건
운영취소 확정 건수: 977건
운영취소 예방률: 54.4%
타 정상 SKU 대체율: 47.9%
운영취소 확정률: 45.6%

[ 취소 사유별 비중 ]
cancel_reason
단순취소        0.545
오집          0.134
타토트오집_공급    0.134
미집          0.063
고객취소_집품후    0.061
토트분실        0.026
리빈파손        0.014
과집          0.013
집품파손        0.010
Name: proportion, dtype: float64


### 성과 정의 (v14)
대체·복구가 필요한 유닛만 KPI 분모에 포함; 과집과 타토트 오집 공급행은 제외.

## **STEP8 - 2단계 분석**

In [9]:
# =============================================
# STEP 8: 2단계 분석 — 개선 전 지연 병 분석
# =============================================

bins   = [0, 10, 20, 30, 50, 70, 100, np.inf]
labels = ['0-10', '10-20', '20-30', '30-50', '50-70', '70-100', '100+']
op_df  = op_df.copy()
op_df['delay_bin'] = pd.cut(
    op_df['process_delay_min'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

delay_summary = op_df.groupby('delay_bin', observed=True)['reused_flag'].mean()

over  = op_df[op_df['process_delay_min'] >  DELAY_THRESHOLD_MIN]['reused_flag'].mean()
under = op_df[op_df['process_delay_min'] <= DELAY_THRESHOLD_MIN]['reused_flag'].mean()

print("=" * 45)
print("2단계. 지연 분석")
print("=" * 45)
print("\n[ 지연 구간별 재사용률 ]")
print(delay_summary.round(3))
print(f"\n30분 이하 재사용률: {under:.1%}")
print(f"30분 초과 재사용률: {over:.1%}  (차이: {under-over:.1%}p)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

delay_summary.plot(kind='bar', ax=axes[0], color='#4C72B0')
axes[0].axvline(x=2.5, color='red', linestyle='--', label='30분 기준선')
axes[0].set_title('지연 구간별 재사용률')
axes[0].set_xlabel('처리 지연 시간(분)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend()

axes[1].scatter(op_df['process_delay_min'], op_df['reused_flag'],
                alpha=0.2, color='#4C72B0')
axes[1].axvline(x=DELAY_THRESHOLD_MIN, color='red', linestyle='--', label='30분')
axes[1].set_title('지연 시간 vs 재사용 여부')
axes[1].set_xlabel('처리 지연 시간(분)')
axes[1].set_ylabel('재사용 여부')
axes[1].legend()

print(op_df['process_delay_min'].describe())
print(op_df['process_delay_min'].min())
print(op_df['process_delay_min'].max())
print(op_df['process_delay_min'].between(0, 10).sum())

plt.tight_layout()
plt.show()

2단계. 지연 분석

[ 지연 구간별 재사용률 ]
delay_bin
0-10     0.500
10-20    0.641
20-30    0.623
30-50    0.355
Name: reused_flag, dtype: float64

30분 이하 재사용률: 53.6%
30분 초과 재사용률: 35.5%  (차이: 18.1%p)
count    2003.000000
mean       11.914184
std        12.255590
min         0.000000
25%         2.211175
50%         4.975746
75%        25.777826
max        37.100967
Name: process_delay_min, dtype: float64
0.0
37.1009668
1247


## **STEP9 - 3단계 분석**

In [10]:
# =============================================
# STEP 9: 3단계 분석 — 사유 분석
# =============================================

reason_reuse  = op_df.groupby('cancel_reason')['reused_flag'].mean()
reason_addcan = op_df.groupby('cancel_reason')['additional_cancel_flag'].mean()

print("=" * 45)
print("3단계. 사유 분석 (운영 문제 후보 기준)")
print("=" * 45)
print("\n[ 사유별 재사용률 ]")
print(reason_reuse.round(3))
print("\n[ 운영취소 확정 사유별 비율 ]")
print(reason_addcan.round(3))

# 오집 상세
wp = op_df[op_df['cancel_reason'] == '오집']
print(f"\n▶ 오집 상세")
print(f"  건수             : {len(wp):,}건")
print(f"  연결된 공급행 검수·등록 통과율: {final_df.loc[final_df['cancel_reason'].eq('타토트오집_공급'),'inspection_flag'].mean():.1%}")
print(f"  재사용률         : {wp['reused_flag'].mean():.1%}")
print(f"  추가 취소 발생률 : {wp['additional_cancel_flag'].mean():.1%}")

# 미집 상세
mp = op_df[op_df['cancel_reason'] == '미집']
print(f"\n▶ 미집 상세")
print(f"  건수               : {len(mp):,}건")
print(f"  대체 재사용률      : {mp['reused_flag'].mean():.1%}")
print(f"  추가 취소 발생률   : {mp['additional_cancel_flag'].mean():.1%}")

# 토트분실 상세
tp = op_df[op_df['cancel_reason'] == '토트분실']
print(f"\n▶ 토트분실 상세")
print(f"  건수               : {len(tp):,}건")
print(f"  타 재고 대체율     : {tp['reused_flag'].mean():.1%}")
print(f"  원 토트 복구율     : {tp['tote_rescue_flag'].mean():.1%}")
print(f"  취소 확정률       : {tp['additional_cancel_flag'].mean():.1%}")

fig, ax = plt.subplots(figsize=(7, 5))

ax.bar(reason_reuse.index, reason_reuse.values)
ax.set_title('사유별 재사용률')
ax.set_xlabel('')
ax.set_ylabel('재사용률')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

3단계. 사유 분석 (운영 문제 후보 기준)

[ 사유별 재사용률 ]
cancel_reason
리빈파손    0.142
미집      0.190
오집      0.752
집품파손    0.170
토트분실    0.072
Name: reused_flag, dtype: float64

[ 운영취소 확정 사유별 비율 ]
cancel_reason
리빈파손    0.858
미집      0.810
오집      0.248
집품파손    0.830
토트분실    0.309
Name: additional_cancel_flag, dtype: float64

▶ 오집 상세
  건수             : 1,163건
  연결된 공급행 검수·등록 통과율: 84.4%
  재사용률         : 75.2%
  추가 취소 발생률 : 24.8%

▶ 미집 상세
  건수               : 547건
  대체 재사용률      : 19.0%
  추가 취소 발생률   : 81.0%

▶ 토트분실 상세
  건수               : 223건
  타 재고 대체율     : 7.2%
  원 토트 복구율     : 61.9%
  취소 확정률       : 30.9%


In [11]:
# =============================================
# STEP 9-3: 추가취소건 사유 비중 분석
# =============================================

# 추가취소 발생 건만 추출
add_cancel_df = op_df[
    op_df['additional_cancel_flag'] == 1
].copy()

# 사유별 건수 집계 (내림차순 정렬)
reason_counts = (
    add_cancel_df['cancel_reason']
    .value_counts()
)

# 사유별 색상 지정
color_map = {
    '오집': '#e76f51',
    '미집': '#f4a261',
    '토트분실': '#9d4edd',
    '집품파손': '#6c757d',
    '리빈파손': '#adb5bd'
}

# 사유 순서에 맞춰 색상 리스트 생성
colors = [
    color_map.get(reason, '#cccccc')
    for reason in reason_counts.index
]

# 파이차트
plt.figure(figsize=(8, 8))

plt.pie(
    reason_counts,
    labels=reason_counts.index,
    autopct='%1.1f%%',
    startangle=90,       # 12시 방향 시작
    counterclock=False,  # 시계방향
    colors=colors
)

plt.title('추가취소건 사유 비중')

plt.tight_layout()
plt.show()

# 건수 출력
print("[추가취소건 사유별 건수]")
print(reason_counts.astype(int))

[추가취소건 사유별 건수]
cancel_reason
미집      443
오집      289
리빈파손    103
집품파손     73
토트분실     69
Name: count, dtype: int64


In [12]:
# =============================================
# STEP 9: 3단계 분석 — 사유 분석
# =============================================

# 재사용 가능 사유만 분석 대상으로 설정
# 파손된 실물은 공급 불가하나 해당 취소건은 다른 정상 SKU 재고를 재사용할 수 있다.
reusable_reasons = OP_REASONS
reason_df = op_df[op_df['cancel_reason'].isin(reusable_reasons)].copy()

# 1) 사유별 재사용률
reason_reuse = (
    reason_df
    .groupby('cancel_reason')['reused_flag']
    .mean()
    .reindex(reusable_reasons)
)

# 2) 추가 취소 사유 비중
# 의미: 추가 취소가 발생한 건 중 각 사유가 차지하는 비중
add_cancel_count = (
    reason_df[reason_df['additional_cancel_flag'] == 1]
    .groupby('cancel_reason')
    .size()
    .reindex(reusable_reasons, fill_value=0)
)

if add_cancel_count.sum() > 0:
    reason_addcan_share = add_cancel_count / add_cancel_count.sum()
else:
    reason_addcan_share = add_cancel_count * 0

print("=" * 45)
print("3단계. 사유 분석 (운영 문제 후보의 대체 재사용 기준)")
print("=" * 45)

print("\n[ 사유별 재사용률 ]")
print(reason_reuse.round(3))

print("\n[ 추가 취소 사유 비중 ]")
print(reason_addcan_share.round(3))

print("\n[ 추가 취소 건수 ]")
print(add_cancel_count)

# 오집 상세
wp = reason_df[reason_df['cancel_reason'] == '오집']
wp_add_count = int(add_cancel_count.get('오집', 0))
wp_add_share = reason_addcan_share.get('오집', 0)

print(f"\n▶ 오집 상세")
print(f"  건수                 : {len(wp):,}건")
print(f"  검수 통과율          : {wp['inspection_flag'].mean():.1%}")
print(f"  재사용률             : {wp['reused_flag'].mean():.1%}")
print(f"  추가 취소 건수       : {wp_add_count:,}건")
print(f"  추가 취소 사유 비중  : {wp_add_share:.1%}")

# 미집 상세
mp = reason_df[reason_df['cancel_reason'] == '미집']
mp_add_count = int(add_cancel_count.get('미집', 0))
mp_add_share = reason_addcan_share.get('미집', 0)

print(f"\n▶ 미집 상세")
print(f"  건수                 : {len(mp):,}건")
print(f"  대체 재사용률        : {mp['reused_flag'].mean():.1%}")
print(f"  추가 취소 건수       : {mp_add_count:,}건")
print(f"  추가 취소 사유 비중  : {mp_add_share:.1%}")

# 토트분실 상세
tp = reason_df[reason_df['cancel_reason'] == '토트분실']
tp_add_count = int(add_cancel_count.get('토트분실', 0))
tp_add_share = reason_addcan_share.get('토트분실', 0)

print(f"\n▶ 토트분실 상세")
print(f"  건수                 : {len(tp):,}건")
print(f"  재사용률             : {tp['reused_flag'].mean():.1%}")
print(f"  추가 취소 건수       : {tp_add_count:,}건")
print(f"  추가 취소 사유 비중  : {tp_add_share:.1%}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

reason_reuse.plot(kind='bar', ax=axes[0], color='#55A868')
axes[0].set_title('사유별 재사용률')
axes[0].set_xlabel('')
axes[0].set_ylim(0, 1)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[0].tick_params(axis='x', rotation=30)

reason_addcan_share.plot(kind='bar', ax=axes[1], color='#C44E52')
axes[1].set_title('추가 취소 사유 비중')
axes[1].set_xlabel('')
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

3단계. 사유 분석 (운영 문제 후보의 대체 재사용 기준)

[ 사유별 재사용률 ]
cancel_reason
오집      0.752
미집      0.190
집품파손    0.170
리빈파손    0.142
토트분실    0.072
Name: reused_flag, dtype: float64

[ 추가 취소 사유 비중 ]
cancel_reason
오집      0.296
미집      0.453
집품파손    0.075
리빈파손    0.105
토트분실    0.071
dtype: float64

[ 추가 취소 건수 ]
cancel_reason
오집      289
미집      443
집품파손     73
리빈파손    103
토트분실     69
dtype: int64

▶ 오집 상세
  건수                 : 1,163건
  검수 통과율          : 0.0%
  재사용률             : 75.2%
  추가 취소 건수       : 289건
  추가 취소 사유 비중  : 29.6%

▶ 미집 상세
  건수                 : 547건
  대체 재사용률        : 19.0%
  추가 취소 건수       : 443건
  추가 취소 사유 비중  : 45.3%

▶ 토트분실 상세
  건수                 : 223건
  재사용률             : 7.2%
  추가 취소 건수       : 69건
  추가 취소 사유 비중  : 7.1%


## **STEP10 - 4단계 분석**

In [13]:
# =============================================
# STEP 10: 4단계 분석 — 카테고리 분석
# =============================================

cat_count  = op_df.groupby('category')['cancel_reason'].count().sort_values(ascending=False)
cat_reuse  = op_df.groupby('category')['reused_flag'].mean().sort_values(ascending=False)
cat_addcan = op_df.groupby('category')['additional_cancel_flag'].mean().sort_values(ascending=False)

print("=" * 45)
print("4단계. 카테고리 분석")
print("=" * 45)
print("\n[ 카테고리별 운영 문제 후보 건수 ]")
print(cat_count)
print("\n[ 카테고리별 재사용률 ]")
print(cat_reuse.round(3))
print("\n[ 카테고리별 추가 취소율 ]")
print(cat_addcan.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cat_count.plot(kind='bar', ax=axes[0], color='#4C72B0')
axes[0].set_title('카테고리별 취소 건수')
axes[0].tick_params(axis='x', rotation=30)

cat_reuse.plot(kind='bar', ax=axes[1], color='#55A868')
axes[1].set_title('카테고리별 재사용률')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].tick_params(axis='x', rotation=30)

cat_addcan.plot(kind='bar', ax=axes[2], color='#C44E52')
axes[2].set_title('카테고리별 추가 취소율')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

4단계. 카테고리 분석

[ 카테고리별 운영 문제 후보 건수 ]
category
가공식품    672
채소      439
우유      367
과일      365
축산      298
Name: cancel_reason, dtype: int64

[ 카테고리별 재사용률 ]
category
채소      0.494
가공식품    0.493
축산      0.483
우유      0.477
과일      0.436
Name: reused_flag, dtype: float64

[ 카테고리별 추가 취소율 ]
category
과일      0.488
축산      0.463
우유      0.455
채소      0.449
가공식품    0.442
Name: additional_cancel_flag, dtype: float64


## **STEP11 — 기존 WMS 조회·취소존 복구·정상 SKU 대체를 구분한 운영취소 예방 비교 (v14)**

In [14]:
# =============================================
# STEP 11: 자동화 도구 vs 수동 — 같은 운영 문제 수요의 운영취소 방지
# =============================================

def complete_scenario(base_df, process_type='manual', tool_usage_rate=1.0):
    # 도구 활용률이 바뀌면 토트 복구, 취소 큐, 공급원, 다른 SKU 대체를 모두 재실행.
    event = build_scenario_pre_match(base_df,process_type,tool_usage_rate)
    event = add_physical_flags(event)
    event = assign_reuse_with_time_window(event,process_type,tool_usage_rate)
    event = add_operational_outcomes(event)
    return event

# STEP 5/6에서 사용한 수동 처리 표본을 비교 기준으로 유지.
before = op_df.copy()
# 자동화는 원본 전체 고객취소 공급원 및 운영 문제 사건으로 재생성.
match_input_df = complete_scenario(base_cancel_df,'automated',BASE_TOOL_USAGE_RATE)
# 재사용 이후 취소 확정된 모든 이벤트(고객취소 포함)가 match_input_df에 들어 있음.
improved_df = match_input_df[match_input_df['cancel_reason'].isin(OP_REASONS)].copy()
search_out = improved_df.apply(calc_search,axis=1)
improved_df['search_start_time']=[r[0] for r in search_out]
improved_df['search_end_time']=[r[1] for r in search_out]
improved_df['search_time_min']=[r[2] for r in search_out]
improved_df['search_wait_min']=[r[3] for r in search_out]
after = improved_df

N_PROBLEMS=len(before)
assert N_PROBLEMS==len(after)
assert set(before['order_id'])==set(after['order_id'])

summary_rows=[]
for label, frame in [('수동',before),('자동',after)]:
    summary_rows.append({
        '시나리오':label,
        '운영_문제_발생':len(frame),
        '원_토트_복구':int(frame['tote_rescue_flag'].sum()),
        '타_재고_대체':int(frame['reused_flag'].sum()),
        '운영취소_예방':int(frame['op_cancel_prevented_flag'].sum()),
        '운영취소_확정':int(frame['op_cancel_confirmed_flag'].sum()),
        '취소_예방률':frame['op_cancel_prevented_flag'].mean(),
        '타_재고_대체율':frame['reused_flag'].mean(),
        '취소_확정률':frame['op_cancel_confirmed_flag'].mean(),
    })
operation_summary_df=pd.DataFrame(summary_rows)
manual_confirmed=int(before['op_cancel_confirmed_flag'].sum())
auto_confirmed=int(after['op_cancel_confirmed_flag'].sum())
op_cancel_reduction_count=manual_confirmed-auto_confirmed
op_cancel_reduction_rate=op_cancel_reduction_count/manual_confirmed if manual_confirmed else float('nan')
print('STEP 11. 동일 운영 문제 후보 사건에 대한 통합 취소 예방 시뮬레이션')
print(operation_summary_df.to_string(index=False,float_format=lambda v:f'{v:.4f}'))
print(f'운영취소 확정 건수: 수동 {manual_confirmed:,}건 → 자동화 {auto_confirmed:,}건')
print(f'기준 대비 운영취소 감소: {op_cancel_reduction_count:+,}건 ({op_cancel_reduction_rate:.2%})')
print('※ 수동/자동 간 운영 대체수요와 고객취소 수는 고정, 원 토트 복구+타 재고 대체는 중복 없음')

print('\n[관리자 대기 한도 검증: 문제 발생시각 기준 피크 20분 / 일반 30분]')
for label,frame in [('수동',before),('자동화',after)]:
    t=frame[frame['cancel_reason'].eq('토트분실')].copy()
    t['피크시간']=t['cancel_time'].dt.hour.isin(PEAK_HOURS)
    t['한도(분)']=(t['tote_cancel_deadline']-t['cancel_time']).dt.total_seconds()/60
    assert t.loc[t['피크시간'],'한도(분)'].eq(TOTE_WAIT_PEAK_MIN).all()
    assert t.loc[~t['피크시간'],'한도(분)'].eq(TOTE_WAIT_NORMAL_MIN).all()
    print(label, t.groupby('피크시간').agg(후보=('order_id','count'),대기한도=('한도(분)','first'),원주문_복구=('tote_rescue_flag','sum')).to_string())

print('\n[기존 WMS 토트 경로 조회 / 취소존 처리 경로 구분]')
for label,frame in [('수동',before),('자동화',after)]:
    t=frame[frame['cancel_reason'].eq('토트분실')]
    print(f'{label}: 기존 경로조회 가능={int(t["tote_existing_route_traced_flag"].sum())}건 '
          f'/ 기존 WMS로 원주문 완료={int(t["tote_existing_route_rescued_flag"].sum())}건 '
          f'/ 취소존 통해 원주문 복구={int((t["tote_rescue_flag"]-t["tote_existing_route_rescued_flag"]).sum())}건 '
          f'/ 취소존 진입 필요={int(t["tote_cancel_zone_required_flag"].sum())}건')
print('\n[토트분실 회수 유형·복구·확정 비교]')
for label,frame in [('수동',before),('자동화',after)]:
    x=frame[frame['cancel_reason'].eq('토트분실')]
    late=x[x['tote_recovery_class'].eq('late_same_day')]
    rare=x[x['tote_recovery_class'].eq('multi_day')]
    eligible_late=late[(late['restock_flag']==1)&(late['inspection_flag']==1)
        &late['tote_return_time'].map(operating_day).eq(late['cancel_time'].map(operating_day))]
    print(f'{label}: 토트분실 유닛={len(x)} / 30분내 회수={int(x["tote_return_within_30_flag"].sum())}'
          f' / 정상 지연회수={len(late)} / 수일 지연={len(rare)}'
          f' / 원토트 복구={int(x["tote_rescue_flag"].sum())}'
          f' / 타재고 대체={int(x["reused_flag"].sum())}'
          f' / 운영취소 확정={int(x["op_cancel_confirmed_flag"].sum())}'
          f' / 지연회수 당일 전산 가용={len(eligible_late)}')

compare=pd.DataFrame({
    '지표':['운영취소 확정 건수','운영취소 예방률','타 재고 대체율','원 토트 복구 건수',
          '대체 수요 조회 준비 평균 지연(분)','취소 처리 후 탐색 평균(분)'],
    '수동 처리 (기존)':[f'{manual_confirmed:,}건',f'{before["op_cancel_prevented_flag"].mean():.1%}',
          f'{before["reused_flag"].mean():.1%}',f'{int(before["tote_rescue_flag"].sum()):,}건',
          f'{before["process_delay_min"].mean():.2f}',f'{before["search_time_min"].mean():.2f}'],
    '자동화 처리 (개선)':[f'{auto_confirmed:,}건',f'{after["op_cancel_prevented_flag"].mean():.1%}',
          f'{after["reused_flag"].mean():.1%}',f'{int(after["tote_rescue_flag"].sum()):,}건',
          f'{after["process_delay_min"].mean():.2f}',f'{after["search_time_min"].mean():.2f}'],
})
print('\n',compare.to_string(index=False))

fig,axes=plt.subplots(1,3,figsize=(15,4.5))
labels=['수동','자동화']
axes[0].bar(labels,[manual_confirmed,auto_confirmed],color=['#C44E52','#55A868'])
axes[0].set_title('최종 운영취소 확정 건수')
axes[0].set_ylabel('건수')
for i,x in enumerate([manual_confirmed,auto_confirmed]):axes[0].text(i,x+8,f'{x:,}',ha='center')
x=np.arange(2)
rescue=[int(before['tote_rescue_flag'].sum()),int(after['tote_rescue_flag'].sum())]
reuse=[int(before['reused_flag'].sum()),int(after['reused_flag'].sum())]
axes[1].bar(x,rescue,label='원 토트 복구',color='#4C72B0')
axes[1].bar(x,reuse,bottom=rescue,label='타 정상 재고 대체',color='#55A868')
axes[1].set_xticks(x,labels);axes[1].set_title('취소 방지 방법 (중복 없음)');axes[1].legend()
axes[2].bar(labels,[before['op_cancel_prevented_flag'].mean(),after['op_cancel_prevented_flag'].mean()],color=['#C44E52','#55A868'])
axes[2].set_title('운영취소 예방률 / 전체 문제 후보');axes[2].set_ylim(0,1)
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_:f'{y:.0%}'))
plt.tight_layout();plt.show()


STEP 11. 동일 운영 문제 후보 사건에 대한 통합 취소 예방 시뮬레이션
시나리오  운영_문제_발생  원_토트_복구  타_재고_대체  운영취소_예방  운영취소_확정  취소_예방률  타_재고_대체율  취소_확정률
  수동      2141      138     1026     1164      977  0.5437    0.4792  0.4563
  자동      2141      155     1160     1315      826  0.6142    0.5418  0.3858
운영취소 확정 건수: 수동 977건 → 자동화 826건
기준 대비 운영취소 감소: +151건 (15.46%)
※ 수동/자동 간 운영 대체수요와 고객취소 수는 고정, 원 토트 복구+타 재고 대체는 중복 없음

[관리자 대기 한도 검증: 문제 발생시각 기준 피크 20분 / 일반 30분]
수동         후보  대기한도  원주문_복구
피크시간                    
False  144  30.0     102
True    79  20.0      36
자동화         후보  대기한도  원주문_복구
피크시간                    
False  144  30.0     113
True    79  20.0      42

[기존 WMS 토트 경로 조회 / 취소존 처리 경로 구분]
수동: 기존 경로조회 가능=181건 / 기존 WMS로 원주문 완료=126건 / 취소존 통해 원주문 복구=12건 / 취소존 진입 필요=97건
자동화: 기존 경로조회 가능=181건 / 기존 WMS로 원주문 완료=126건 / 취소존 통해 원주문 복구=29건 / 취소존 진입 필요=97건

[토트분실 회수 유형·복구·확정 비교]
수동: 토트분실 유닛=223 / 30분내 회수=205 / 정상 지연회수=16 / 수일 지연=2 / 원토트 복구=138 / 타재고 대체=16 / 운영취소 확정=69 / 지연회수 당일 전산 가용=14
자동화: 토트분실 유닛=223 / 30분내 회수=205 / 정상 

### STEP 11-B — 토트별 10~20 계획 유닛 및 상품별 부분 취소
인접 토트는 유사 시간대 내 독립 집품 완료, 같은 토트 안에서 일반 이송시각 공유. 실제 토트 전체 분실 및 실제 물리 적재 원장 미구현.

In [15]:
# STEP 11-B. 원본 150,000 유닛과 운영 문제 결과를 합쳐 토트별 부분 취소를 검증
# 정상 유닛은 문제가 없다면 정상 완료, 고객취소는 운영 취소와 별도로 분류.
unit_catalog=orders_df[['order_id','unit_id','tote_id','tote_unit_no','tote_units_total']].copy()
def tote_outcomes(frame):
    x=unit_catalog.merge(frame[['order_id','cancel_reason','tote_rescue_flag','reused_flag',
        'op_cancel_confirmed_flag']],on='order_id',how='left',validate='one_to_one')
    op=x['cancel_reason'].isin(OP_REASONS)
    customer=x['cancel_reason'].isin(['단순취소','고객취소_집품후'])
    x['unit_outcome']='정상완료'
    x.loc[customer,'unit_outcome']='고객취소'
    x.loc[op,'unit_outcome']='운영취소_확정'
    x.loc[op & x['reused_flag'].eq(1),'unit_outcome']='정상재고_대체'
    x.loc[op & x['tote_rescue_flag'].eq(1),'unit_outcome']='원토트_복구'
    x['op_confirmed_unit']=x['unit_outcome'].eq('운영취소_확정').astype(int)
    x['completed_unit']=x['unit_outcome'].isin(['정상완료','정상재고_대체','원토트_복구']).astype(int)
    agg=x.groupby('tote_id').agg(total_units=('unit_id','size'),
        op_confirmed_units=('op_confirmed_unit','sum'),completed_units=('completed_unit','sum'))
    agg['partial_op_cancel_tote']=((agg['op_confirmed_units']>0)&(agg['completed_units']>0)).astype(int)
    agg['all_op_cancel_tote']=(agg['op_confirmed_units']==agg['total_units']).astype(int)
    assert agg['total_units'].between(TOTE_CAPACITY_MIN,TOTE_CAPACITY_MAX).all()
    assert int(agg['op_confirmed_units'].sum())==int(frame['op_cancel_confirmed_flag'].sum())
    assert len(x)==N_ORDERS
    return x,agg
manual_tote_units_df,manual_tote_summary_df=tote_outcomes(before)
auto_tote_units_df,auto_tote_summary_df=tote_outcomes(after)
print('토트별 집품 구조 / 상품별 일부 취소 (시뮬레이션)')
print(pd.DataFrame([{'처리방식':'수동','토트수':len(manual_tote_summary_df),
    '부분 운영취소 토트수':int(manual_tote_summary_df['partial_op_cancel_tote'].sum()),
    '확정 취소 유닛':int(manual_tote_summary_df['op_confirmed_units'].sum())},
    {'처리방식':'자동','토트수':len(auto_tote_summary_df),
    '부분 운영취소 토트수':int(auto_tote_summary_df['partial_op_cancel_tote'].sum()),
    '확정 취소 유닛':int(auto_tote_summary_df['op_confirmed_units'].sum())}]).to_string(index=False))
print('유닛수 분포:',tote_sizes.value_counts().sort_index().to_dict() if 'tote_sizes' in globals() else orders_df.groupby('tote_id').size().value_counts().sort_index().to_dict())
example_tote=auto_tote_summary_df.index[auto_tote_summary_df['partial_op_cancel_tote'].eq(1)][:1]
if len(example_tote):
    print('부분 운영취소 토트 예시:',example_tote.tolist())
    print(auto_tote_units_df[auto_tote_units_df['tote_id'].isin(example_tote)][['tote_id','unit_id','unit_outcome']].to_string(index=False))


토트별 집품 구조 / 상품별 일부 취소 (시뮬레이션)
처리방식   토트수  부분 운영취소 토트수  확정 취소 유닛
  수동 10001          932       977
  자동 10001          790       826
유닛수 분포: {10: 917, 11: 850, 12: 937, 13: 914, 14: 940, 15: 904, 16: 906, 17: 936, 18: 895, 19: 924, 20: 878}
부분 운영취소 토트 예시: ['TOTE-000003']
    tote_id     unit_id unit_outcome
TOTE-000003 UNIT-053218         정상완료
TOTE-000003 UNIT-055426         정상완료
TOTE-000003 UNIT-056297      운영취소_확정
TOTE-000003 UNIT-056597         정상완료
TOTE-000003 UNIT-057454         정상완료
TOTE-000003 UNIT-058862         정상완료
TOTE-000003 UNIT-059098         정상완료
TOTE-000003 UNIT-061064         정상완료
TOTE-000003 UNIT-063305         정상완료
TOTE-000003 UNIT-064537         정상완료
TOTE-000003 UNIT-064735         정상완료
TOTE-000003 UNIT-065419         정상완료
TOTE-000003 UNIT-068325         정상완료
TOTE-000003 UNIT-068553         정상완료
TOTE-000003 UNIT-073452         정상완료
TOTE-000003 UNIT-077230         정상완료
TOTE-000003 UNIT-080698         정상완료
TOTE-000003 UNIT-082539         정상완료
TOTE-000003 UNIT-083840   

**이전 버전 주의:** 최신 수치는 STEP 11과 본 README 기준. 과집을 수요에서 제외해 분모가 변경됨.

**과거 설계 메모(참고용):** 최종 기준은 STEP 5, STEP 11-D/E, STEP 12 및 README를 참조하세요.

**과거 버전 결과 (원본 노트북 보관):** 이 셀의 이전 고정 KPI는 현재 재사용/공급원 규칙과 직접 비교 불가하여 새본에서는 실행하지 않습니다.

**과거 설계 메모(참고용):** 최종 기준은 STEP 5, STEP 11-D/E, STEP 12 및 README를 참조하세요.

**과거 버전 결과 (원본 노트북 보관):** 이 셀의 이전 고정 KPI는 현재 재사용/공급원 규칙과 직접 비교 불가하여 새본에서는 실행하지 않습니다.

**과거 버전 결과 (원본 노트북 보관):** 이 셀의 이전 고정 KPI는 현재 재사용/공급원 규칙과 직접 비교 불가하여 새본에서는 실행하지 않습니다.

**과거 설계 메모(참고용):** 최종 기준은 STEP 5, STEP 11-D/E, STEP 12 및 README를 참조하세요.

In [16]:
# =============================================
# STEP 11-3: 당일 기존재고 우선 + 미래 30분 대기 매칭률 비교
# =============================================

target_reasons = OP_REASONS

before_match_df = before[before['cancel_reason'].isin(target_reasons)].copy()
after_match_df = after[after['cancel_reason'].isin(target_reasons)].copy()

match_compare = pd.DataFrame({
    '개선 전 매칭 가능률': before_match_df.groupby('cancel_reason')['time_window_match_flag'].mean(),
    '개선 후 매칭 가능률': after_match_df.groupby('cancel_reason')['time_window_match_flag'].mean(),
    '개선 전 타 재고 대체율': before_match_df.groupby('cancel_reason')['reused_flag'].mean(),
    '개선 후 타 재고 대체율': after_match_df.groupby('cancel_reason')['reused_flag'].mean(),
})

print('=' * 68)
print('STEP 11-3. 당일 기존재고 우선 + 미래 30분 대기, 타 재고 대체 비교')
print('=' * 68)
print(match_compare.round(3))

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(match_compare.index))
width = 0.35
ax.bar(x - width/2, match_compare['개선 전 매칭 가능률'], width, label='개선 전 매칭 가능률')
ax.bar(x + width/2, match_compare['개선 후 매칭 가능률'], width, label='개선 후 매칭 가능률')
ax.set_title('당일 기존재고 우선 + 미래 30분 대기 매칭률 비교')
ax.set_ylabel('매칭률')
ax.set_xticks(x)
ax.set_xticklabels(match_compare.index)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()


STEP 11-3. 당일 기존재고 우선 + 미래 30분 대기, 타 재고 대체 비교
               개선 전 매칭 가능률  개선 후 매칭 가능률  개선 전 타 재고 대체율  개선 후 타 재고 대체율
cancel_reason                                                        
리빈파손                 0.150        0.142          0.142          0.142
미집                   0.221        0.190          0.190          0.188
오집                   0.863        0.868          0.752          0.865
집품파손                 0.193        0.205          0.170          0.205
토트분실                 0.085        0.072          0.072          0.072


**과거 설계 메모(참고용):** 최종 기준은 STEP 5, STEP 11-D/E, STEP 12 및 README를 참조하세요.

## STEP 11-D/E — 검색툴 사용률·SKU 분포 민감도 (v14 통합 취소 예방)

도구 사용률이 달라지면 **취소 전 토트 복구 건수, 취소처리 큐, 공급 가능 재고, 타 주문 재사용이 함께 변하므로 전 과정을 재계산**합니다.
기존 토트 경로 조회 80%는 검색툴 활용률 민감도와 무관하게 양쪽에 고정한다.


In [17]:
# =============================================
# STEP 11-D. 검색 수행률 민감도(70/80/90/100%) — 취소전 복구 + 취소후 대체 통합
# =============================================
tool_usage_sensitivity_rows=[]
for rate in (0.70,0.80,0.90,1.00):
    _whole=build_scenario_pre_match(base_cancel_df,'automated',rate)
    _whole=add_physical_flags(_whole)
    _whole=assign_reuse_with_time_window(_whole,'automated',rate)
    _whole=add_operational_outcomes(_whole)
    _op=_whole[_whole['cancel_reason'].isin(OP_REASONS)]
    _confirmed=int(_op['op_cancel_confirmed_flag'].sum())
    tool_usage_sensitivity_rows.append({
        '도구_검색_수행률_가정':rate,
        '실현_검색_수행률':_op['tool_used_flag'].mean(),
        '토트_원주문_복구':int(_op['tote_rescue_flag'].sum()),
        '타_재고_대체율':_op['reused_flag'].mean(),
        '운영취소_확정':_confirmed,
        '수동대비_취소감소':manual_confirmed-_confirmed,
        '수동대비_취소감소율':(manual_confirmed-_confirmed)/manual_confirmed,
    })
tool_usage_sensitivity_df=pd.DataFrame(tool_usage_sensitivity_rows)
print('STEP 11-D 검색툴 수행률 민감도 (실측 아님; 원 토트 복구부터 재시뮬레이션)')
print(tool_usage_sensitivity_df.round(4).to_string(index=False))
assert int(tool_usage_sensitivity_df.iloc[-1]['운영취소_확정'])==auto_confirmed


STEP 11-D 검색툴 수행률 민감도 (실측 아님; 원 토트 복구부터 재시뮬레이션)
 도구_검색_수행률_가정  실현_검색_수행률  토트_원주문_복구  타_재고_대체율  운영취소_확정  수동대비_취소감소  수동대비_취소감소율
          0.7     0.6436        151    0.5283      859        118      0.1208
          0.8     0.7305        151    0.5325      850        127      0.1300
          0.9     0.8403        154    0.5367      838        139      0.1423
          1.0     0.9411        155    0.5418      826        151      0.1546


In [18]:
# =============================================
# STEP 11-E. SKU 종류·주문 집중도 민감도 (동일 취소 사건, 고정 난수)
# =============================================
# 이 실험의 상품 ID 재배치는 '가상 상품 수요'를 바꾸는 것으로 실제 SKU 분포를 추정하지 않는다.
# 업무 사건(취소시각·사유·도착·검수)과 manual/auto 비교 구조는 공통으로 유지한다.
# 가상 SKU 마스터가 달라지므로 경제가치 분석은 이 민감도 결과에 적용하지 않는다.
def create_sku_scenario(df, n_products=500, concentrated=False):
    s = df.copy()
    u_prod = s['error_event_id'].map(lambda oid: deterministic_uniform(oid, 'scenario_sku_product'))
    # u²로 소수 상품에 주문이 집중되는 임의 시나리오(실제 수요 추정 아님).
    ranks = (u_prod**2 if concentrated else u_prod)
    ids = np.minimum((ranks*n_products).astype(int)+1, n_products)
    s['product_id'] = [f'PRD-{p:04d}' for p in ids]
    u_variant = s['error_event_id'].map(lambda oid: deterministic_uniform(oid, 'scenario_sku_variant'))
    s['sku_id'] = [f'{p}-SKU-{int(u*3)+1:02d}' for p,u in zip(s['product_id'],u_variant)]
    return s

sku_sensitivity_rows = []
for sku_count in (250, 500, 1000):
    for concentrated in (False,True):
        _s_manual = create_sku_scenario(final_df,sku_count,concentrated)
        _s_auto = create_sku_scenario(match_input_df,sku_count,concentrated)
        _bm = assign_reuse_with_time_window(_s_manual,'manual')
        _am = assign_reuse_with_time_window(_s_auto,'automated',tool_usage_rate=1.0)
        _bm = add_operational_outcomes(_bm)
        _bm = _bm[_bm['cancel_reason'].isin(OP_REASONS)]
        _am = add_operational_outcomes(_am)
        _am = _am[_am['cancel_reason'].isin(OP_REASONS)]
        assert set(_bm['order_id']) == set(_am['order_id'])
        sku_sensitivity_rows.append({
            '상품_종류_수':sku_count,
            '분포': '집중형(u²)' if concentrated else '균등형(u)',
            '수동_매칭가능률':_bm['time_window_match_flag'].mean(),
            '자동_매칭가능률':_am['time_window_match_flag'].mean(),
            '수동_재사용률':_bm['reused_flag'].mean(),
            '자동_재사용률':_am['reused_flag'].mean(),
            '개선_pp':100*(_am['reused_flag'].mean()-_bm['reused_flag'].mean()),
            '수동_운영취소_확정':int(_bm['op_cancel_confirmed_flag'].sum()),
            '자동_운영취소_확정':int(_am['op_cancel_confirmed_flag'].sum()),
            '운영취소_감소':int(_bm['op_cancel_confirmed_flag'].sum()-_am['op_cancel_confirmed_flag'].sum())
        })
sku_sensitivity_df = pd.DataFrame(sku_sensitivity_rows)
print('STEP 11-E SKU 수 / 수요 집중도 민감도: 원 토트 복구를 포함한 최종 운영취소 건수')
print(sku_sensitivity_df.round(4).to_string(index=False))


STEP 11-E SKU 수 / 수요 집중도 민감도: 원 토트 복구를 포함한 최종 운영취소 건수
 상품_종류_수      분포  수동_매칭가능률  자동_매칭가능률  수동_재사용률  자동_재사용률  개선_pp  수동_운영취소_확정  자동_운영취소_확정  운영취소_감소
     250  균등형(u)    0.6109    0.5862   0.5241   0.5829 5.8851         881         738      143
     250 집중형(u²)    0.6324    0.6011   0.5409   0.5988 5.7917         845         704      141
     500  균등형(u)    0.5474    0.5353   0.4727   0.5334 6.0719         991         844      147
     500 집중형(u²)    0.5759    0.5563   0.4960   0.5544 5.8384         941         799      142
    1000  균등형(u)    0.5054    0.5012   0.4395   0.4998 6.0252        1062         916      146
    1000 집중형(u²)    0.5404    0.5283   0.4666   0.5264 5.9785        1004         859      145


## STEP 12-1 — 확장 시뮬레이션: 회수 실물의 재진열 지연 감가

In [19]:
# =============================================
# STEP 12-1. 확장 분석: 회수 정상 상품의 '재진열 지연 추가 감가'
# =============================================
# 판매실패 확률 / 매출전액 보존 / 인건비·ROI / 미집·파손 실물 가치는 반영하지 않는다.
# 실물 회수·검수 통과 공급원 1개 = 경제 계산 1개. 운영취소 건과 재고 공급원은 분리.
# 공급 실물의 소비 여부를 주문 ID로 추적해 이미 재사용된 실물에는 재진열 감가 0원을 적용.
# 손실 = 정상 즉시 가용일 할인 대비 재진열 판매기준일 할인(유통기한 경과 시 폐기).
# 상품 자체의 재고 가치 증분에 대한 조건부 모형으로, 기업 순비용 절감/정상 재고 대체 기회비용 아님.
import hashlib

def stable_uniform(key, salt=''):
    return deterministic_uniform(key, salt)

def stable_hour_in_window(order_id, start_hour, end_hour, salt):
    u = stable_uniform(order_id,salt)
    offset = min(int(u*(end_hour-start_hour)*60),(end_hour-start_hour)*60-1)
    return start_hour,offset

def get_discount_rate(category, remaining_days):
    if remaining_days < 0:return 1.0  # 유통기한 경과: 폐기
    for min_days, rate in DISCOUNT_RULES_BY_CATEGORY.get(category,[(0, .3)]):
        if remaining_days >= min_days:return rate
    return .3

def get_redisplay_schedule(row):
    # 기존 상품군별 재진열 스케줄 가정 유지. 최소 상품 회수 후에만 진열 가능.
    event = pd.Timestamp(row['cancel_time'])
    availability = pd.Timestamp(row['physical_available_time'])
    baseline = event.normalize()
    same_day_late = (event.hour < MORNING_SHIFT_END_HOUR and
        stable_uniform(row['order_id'],'same_day_late') < MORNING_SAME_DAY_REDISPLAY_PROB)
    if same_day_late:
        date, start, end, salt = baseline,21,24,'late_window'
    else:
        date, start, end, salt = baseline+pd.Timedelta(days=1),12,17,'afternoon_window'
    h,m = stable_hour_in_window(row['order_id'],start,end,salt)
    redisplay = date + pd.Timedelta(hours=h,minutes=m)
    readiness = max(availability, pd.Timestamp(row['cancel_zone_processed_at'])) if row['cancel_reason']=='고객취소_집품후' and pd.notna(row['cancel_zone_processed_at']) else availability
    if redisplay <= readiness:
        date = readiness.normalize() + pd.Timedelta(days=1)
        h,m = stable_hour_in_window(row['order_id'],12,17,'rollover_window')
        redisplay = date + pd.Timedelta(hours=h,minutes=m)
    sale_date = redisplay.normalize() + (pd.Timedelta(days=1) if redisplay.hour>=21 else pd.Timedelta(0))
    return redisplay,sale_date

def recovered_sources(df):
    recovered = df[(df['cancel_reason'].isin(SUPPLY_REASONS))
        & (df['physical_inspection_pass_flag']==1)
        & (df['source_assignment_released_flag']==1)].copy()
    recovered['physical_available_time'] = recovered['process_time']
    tm = recovered['cancel_reason'].eq('토트분실')
    recovered.loc[tm,'physical_available_time'] = recovered.loc[tm,['tote_return_time','process_time','queue_entry_time']].max(axis=1)
    cm = recovered['cancel_reason'].eq('고객취소_집품후')
    recovered.loc[cm,'physical_available_time'] = recovered.loc[cm,'customer_cancel_return_time']
    recovered = recovered[recovered['physical_available_time'].notna()].copy()
    # 기준 운영일 회수분만 계산: 수일 후 발견된 1% 극단 지연 토트는 당일 실물 공급원 아님.
    same_day=recovered['physical_available_time'].map(operating_day).eq(
        recovered['cancel_time'].map(operating_day))
    return recovered.loc[same_day].copy()

def build_markdown_detail(df):
    source = recovered_sources(df)
    consumed = set(df['consumed_candidate_order_id'].dropna())
    assert df['consumed_candidate_order_id'].dropna().is_unique  # 1개 재고의 다중 소비 금지
    source['consumed'] = source['order_id'].isin(consumed)
    source['source_discount'] = source.apply(lambda r:get_discount_rate(
        r['category'],(pd.Timestamp(r['expiry_date']).normalize()-pd.Timestamp(r['physical_available_time']).normalize()).days),axis=1)
    sched = source.apply(get_redisplay_schedule,axis=1)
    source['redisplay_time'] = [x[0] for x in sched]
    source['redisplay_sale_date'] = [x[1] for x in sched]
    source['redisplay_discount'] = source.apply(lambda r:get_discount_rate(
        r['category'],(pd.Timestamp(r['expiry_date']).normalize()-r['redisplay_sale_date']).days),axis=1)
    source['extra_discount_rate'] = (source['redisplay_discount']-source['source_discount']).clip(lower=0)
    # 공급되었으면 재진열이 발생하지 않음. 감가 계산 0원.
    source['markdown_loss'] = np.where(source['consumed'],0,
        source['extra_discount_rate']*source['product_price'])
    source['markdown_loss'] = source['markdown_loss'].clip(lower=0)
    return source

# 개선 전: 전체 수동 데이터의 원본 공급원 + 수동 소비 이력
before_value_df = build_markdown_detail(final_df)
# 개선 후: 고객취소 공급원까지 포함한 재사용 판정 결과
# match_input_df는 STEP 11 최종 매칭 결과를 보유해야 함.
after_value_df = build_markdown_detail(match_input_df)
# v5에서는 원 토트 복구로 자동화에만 제외되는 공급원이 생길 수 있다.
# 두 시나리오가 동일 공급원 집합이어야 한다는 v4 가정은 더 이상 성립하지 않는다.
assert (before_value_df.loc[before_value_df['consumed'],'markdown_loss']==0).all()
assert (after_value_df.loc[after_value_df['consumed'],'markdown_loss']==0).all()
assert not before_value_df['cancel_reason'].eq('미집').any()
assert not after_value_df['cancel_reason'].eq('미집').any()

baseline_markdown = before_value_df['markdown_loss'].sum()
automated_markdown = after_value_df['markdown_loss'].sum()
delta_markdown = baseline_markdown-automated_markdown
print('STEP 12-1 재진열 지연에 따른 추가 감가 (판매실패 확률 미적용)')
print('수동 회수·검수 정상 공급원:',len(before_value_df),' / 자동:',len(after_value_df))
print('수동 재사용된 공급원 수:',int(before_value_df['consumed'].sum()))
print('자동 재사용된 공급원 수:',int(after_value_df['consumed'].sum()))
print(f'수동 잔여 재진열 지연 감가: {baseline_markdown:,.0f}원')
print(f'자동 잔여 재진열 지연 감가: {automated_markdown:,.0f}원')
print(f'조건부 감가 차이(수동-자동): {delta_markdown:,.0f}원')
print('※ 정상 재고 대체의 기회비용을 제외한 회수상품의 부분 가치 시나리오이며 기업 순절감액이 아닙니다.')


STEP 12-1 재진열 지연에 따른 추가 감가 (판매실패 확률 미적용)
수동 회수·검수 정상 공급원: 1677  / 자동: 1660
수동 재사용된 공급원 수: 1026
자동 재사용된 공급원 수: 1160
수동 잔여 재진열 지연 감가: 60,200원
자동 잔여 재진열 지연 감가: 43,890원
조건부 감가 차이(수동-자동): 16,310원
※ 정상 재고 대체의 기회비용을 제외한 회수상품의 부분 가치 시나리오이며 기업 순절감액이 아닙니다.


## STEP 12-2 — 연간 가정 시나리오: 공급원 합집합 재표본화

원 토트 복구 건은 취소처리 공급원으로 투입되지 않으므로, 수동·자동화별 공급원은 다를 수 있습니다. 양쪽 공급원의 합집합으로 금액 차이를 계산하고 해당 시나리오에 공급되지 않은 상품의 추가 재진열 감가를 0으로 둡니다. 월별 운영취소 확정 건수를 추정한 것이 아니라 **운영 문제 후보 규모가 반복된다는 가정**입니다.

In [20]:
# =============================================
# STEP 12-2. 공급원 실물의 짝지은 재진열 감가 재표본화 (연간 가정 시나리오)
# =============================================
# 이 표본은 1운영일의 정상 회수상품 구성 반복이다. 주문 증가·계절성·새 매칭 재시뮬레이션 아님.
base_daily_orders = N_ORDERS
month_days = [31,28,31,30,31,30,31,31,30,31,30,31]
quarterly_daily_order_increase = 20000
monthly_daily_growth = quarterly_daily_order_increase/3
ORDER_VOLUME_ERROR_RATE = 0.05
op_cancel_rate = CANCEL_RATE * CANCEL_TYPE_WEIGHTS['운영취소']
# 두 시나리오가 같은 원 토트 공급원을 갖지 않을 수 있음: 원 주문 복구한 건은 재진열감가 0원.
# 주문 ID의 합집합을 사용해 해당 경로에서 공급원이 아닌 경우 0원으로 계산한다.
paired_value_base = before_value_df[['order_id','markdown_loss']].rename(
    columns={'markdown_loss':'manual_markdown'}).merge(
    after_value_df[['order_id','markdown_loss']].rename(columns={'markdown_loss':'automated_markdown'}),
    on='order_id',how='outer',validate='one_to_one').fillna(0)
assert paired_value_base['order_id'].is_unique
# 기준일 운영 문제 후보 1건당 회수 정상 공급원 수의 합집합 비율(집품후 고객취소 포함)
donor_per_op_event = len(paired_value_base)/int(original_cancel_df['cancel_reason'].isin(OP_REASONS).sum())  # 원래 오류 표본 비율에 대응
rng = np.random.default_rng(1202)
monthly_rows = []
for month,ndays in enumerate(month_days,1):
    expected_orders = base_daily_orders+(month-1)*monthly_daily_growth
    month_orders = int(expected_orders*rng.uniform(.95,1.05))
    month_op_count = 0
    manual_total, auto_total = 0.0,0.0
    for day in range(ndays):
        daily_orders = int(month_orders*rng.uniform(.95,1.05))
        op_count = max(1,int(daily_orders*op_cancel_rate))
        # 표본 수를 재고 공급원 비율로 확대. 운영취소 1건당 공급원 수가 고정이라는 가정.
        donor_count = max(1,int(round(op_count*donor_per_op_event)))
        s = paired_value_base.iloc[rng.integers(0,len(paired_value_base),size=donor_count)]
        manual_total += s['manual_markdown'].sum()
        auto_total += s['automated_markdown'].sum()
        month_op_count += op_count
    monthly_rows.append({'month':month,'simulated_op_cancel_count':month_op_count,
        'simulated_recovered_goods_count':int(round(month_op_count*donor_per_op_event)),
        'manual_markdown':manual_total,'automated_markdown':auto_total,
        'markdown_difference':manual_total-auto_total})
monthly_value_check_df = pd.DataFrame(monthly_rows)
monthly_value_check_df['markdown_difference_per_op'] = (monthly_value_check_df['markdown_difference']/
    monthly_value_check_df['simulated_op_cancel_count'])
print('STEP 12-2 월별 조건부 재진열 감가 (상품구성 재표본화, 실제 월별 예측 아님)')
print(monthly_value_check_df.round(2).to_string(index=False))


STEP 12-2 월별 조건부 재진열 감가 (상품구성 재표본화, 실제 월별 예측 아님)
 month  simulated_op_cancel_count  simulated_recovered_goods_count  manual_markdown  automated_markdown  markdown_difference  markdown_difference_per_op
     1                      66257                            49296        1808110.0           1296080.0             512030.0                        7.73
     2                      68040                            50622        1794750.0           1296230.0             498520.0                        7.33
     3                      76268                            56744        2052040.0           1487300.0             564740.0                        7.40
     4                      73667                            54809        2056850.0           1485410.0             571440.0                        7.76
     5                      81623                            60728        2316150.0           1669140.0             647010.0                        7.93
     6                      83715

## STEP 12-3 — 연간 감가 차이 (실제 순절감액 아님)

In [21]:
# =============================================
# STEP 12-3. 연간 감가 차이 참고 시나리오 (메인 성과와 분리)
# =============================================
annual_op_cancel = int(monthly_value_check_df['simulated_op_cancel_count'].sum())
annual_manual_markdown = monthly_value_check_df['manual_markdown'].sum()
annual_auto_markdown = monthly_value_check_df['automated_markdown'].sum()
annual_markdown_difference = annual_manual_markdown-annual_auto_markdown
annual_reduction_rate = (annual_markdown_difference/annual_manual_markdown
                         if annual_manual_markdown > 0 else float('nan'))
print(f'연간 가정 운영 문제 후보 이벤트: {annual_op_cancel:,}건')
print(f'연간 수동 재진열 지연 감가 노출: {annual_manual_markdown:,.0f}원')
print(f'연간 자동 재진열 지연 감가 노출: {annual_auto_markdown:,.0f}원')
print(f'연간 조건부 감가 차이: {annual_markdown_difference:,.0f}원')
print(f'감가 차이 비율(수동 기준): {annual_reduction_rate:.2%}')
print('※ 판매실패·인건비·기존 정상재고 대체의 기회비용·도입비용을 제외한 부분가치 시나리오입니다.')
print('※ 기준일 공급원 구성과 재사용/소비 판정을 월마다 복원추출하므로 주문량 변화에 따른 대기열 재연산은 하지 않습니다.')


연간 가정 운영 문제 후보 이벤트: 1,023,536건
연간 수동 재진열 지연 감가 노출: 27,455,270원
연간 자동 재진열 지연 감가 노출: 19,937,790원
연간 조건부 감가 차이: 7,517,480원
감가 차이 비율(수동 기준): 27.38%
※ 판매실패·인건비·기존 정상재고 대체의 기회비용·도입비용을 제외한 부분가치 시나리오입니다.
※ 기준일 공급원 구성과 재사용/소비 판정을 월마다 복원추출하므로 주문량 변화에 따른 대기열 재연산은 하지 않습니다.


### 분석 해석 메모 (최신)

STEP 12 경제효과(v14)는 원 토트 복구로 공급원이 사라지는 건을 시나리오별로 분리하며, **실제 원가절감 또는 기업 순이익이 아닌**, 회수 상품이 다시 진열되면서 **당초 가용일 대비 유통기한 할인 구간이 악화되어 발생하는 추가 감가**에 관한 조건부 계산입니다. 판매실패확률은 제거했고, 재고 소비 1:1 추적으로 중복 계산을 차단했습니다. 기존 버전의 8.05%, 9.48%, 2억 원 등은 최신 결과가 아니므로 재사용하지 않습니다.


## **STEP13 - 저장**

In [22]:
print('[로컬 검증] Google Drive 저장 셀은 실행 생략')

[로컬 검증] Google Drive 저장 셀은 실행 생략


**시뮬레이션 데이터 기반이며 실제 운영 데이터 적용 시 수치는 달라질 수 있다**

# **운영데이터 흐름 개선용 분석**

## STEP 14 — 확장 아이디어: 시간·구역별 오집/미집 집중 구간 탐색 (실제 고위험 탐지 아님)

In [23]:
# =============================================
# STEP 14: 1시간 단위 오집·미집 이상 로케이션 탐지
# - 기존 final_df를 변경하지 않고 확장 분석용 데이터 생성
# =============================================

hourly_df = improved_df.copy()

# 운영취소만 분석
hourly_df = hourly_df[~hourly_df['cancel_reason'].isin(['단순취소', '고객취소_집품후'])].copy()

# 오집/미집 여부
hourly_df['error_flag'] = hourly_df['cancel_reason'].isin(['오집', '미집']).astype(int)
hourly_df['wrong_pick_flag'] = (hourly_df['cancel_reason'] == '오집').astype(int)
hourly_df['missing_pick_flag'] = (hourly_df['cancel_reason'] == '미집').astype(int)

# 로케이션 구역
hourly_df['stock_zone'] = (
    hourly_df['stock_location']
    .astype(str)
    .str.split('-')
    .str[0]
)

# 시간 단위 구간 생성
hourly_df['cancel_hour'] = hourly_df['cancel_time'].dt.floor('h')  # [수정] pandas 2.2+ 호환: 'H' → 'h'

print("=" * 60)
print("STEP 14. 1시간 단위 오집·미집 이상 로케이션 탐지")
print("=" * 60)
print(f"운영 문제 후보 건수: {len(hourly_df):,}건")
print(f"오집·미집 건수: {hourly_df['error_flag'].sum():,}건")
print(f"분석 시간 구간 수: {hourly_df['cancel_hour'].nunique():,}개")
print(f"분석 로케이션 구역 수: {hourly_df['stock_zone'].nunique():,}개")
print('※ 교육 아이디어는 타토트 오집·부족집품 수요만 집계. 과집 공급 사건 및 재고 흐름 재시뮬레이션 제외.')

STEP 14. 1시간 단위 오집·미집 이상 로케이션 탐지
운영 문제 후보 건수: 2,141건
오집·미집 건수: 1,710건
분석 시간 구간 수: 17개
분석 로케이션 구역 수: 5개
※ 교육 아이디어는 타토트 오집·부족집품 수요만 집계. 과집 공급 사건 및 재고 흐름 재시뮬레이션 제외.


In [24]:
# =============================================
# STEP 14-1: 시간 × 로케이션별 오집·미집 집계
# =============================================

hour_location_summary = (
    hourly_df
    .groupby(['cancel_hour', 'stock_zone'])
    .agg(
        total_op_cancel=('order_id', 'count'),
        error_count=('error_flag', 'sum'),
        wrong_pick_count=('wrong_pick_flag', 'sum'),
        missing_pick_count=('missing_pick_flag', 'sum'),
        additional_cancel_count=('additional_cancel_flag', 'sum'),
        additional_cancel_rate=('additional_cancel_flag', 'mean')
    )
    .reset_index()
)

hour_location_summary['error_rate'] = (
    hour_location_summary['error_count'] /
    hour_location_summary['total_op_cancel']
)

# 시간대별 평균/표준편차 대비 이상치 판단
hour_stats = (
    hour_location_summary
    .groupby('cancel_hour')
    .agg(
        hour_avg_error_count=('error_count', 'mean'),
        hour_std_error_count=('error_count', 'std'),
        hour_avg_error_rate=('error_rate', 'mean'),
        hour_std_error_rate=('error_rate', 'std')
    )
    .reset_index()
)

hour_location_summary = hour_location_summary.merge(
    hour_stats,
    on='cancel_hour',
    how='left'
)

# 표준편차가 0 또는 NaN인 경우 보정
hour_location_summary['hour_std_error_count'] = (
    hour_location_summary['hour_std_error_count']
    .fillna(0)
)

hour_location_summary['hour_std_error_rate'] = (
    hour_location_summary['hour_std_error_rate']
    .fillna(0)
)

# 이상치 기준:
# 1. 해당 시간 평균 오류 건수보다 많고
# 2. 평균 + 1표준편차 이상이거나, 운영취소 내 사유비중이 시간 평균보다 높고
# 3. 최소 오류 건수 2건 이상
MIN_HOURLY_ERROR_COUNT = 2

hour_location_summary['is_outlier_location'] = (
    (
        hour_location_summary['error_count'] >=
        hour_location_summary['hour_avg_error_count'] + hour_location_summary['hour_std_error_count']
    ) |
    (
        hour_location_summary['error_rate'] >=
        hour_location_summary['hour_avg_error_rate'] + hour_location_summary['hour_std_error_rate']
    )
) & (
    hour_location_summary['error_count'] >= MIN_HOURLY_ERROR_COUNT
)

outlier_locations = hour_location_summary[
    hour_location_summary['is_outlier_location']
].copy()

outlier_locations = outlier_locations.sort_values(
    ['cancel_hour', 'error_count', 'error_rate'],
    ascending=[True, False, False]
)

print("=" * 60)
print("시간별 이상 로케이션 탐지 결과")
print("=" * 60)
print(f"이상 로케이션 감지 건수: {len(outlier_locations):,}건")

print(outlier_locations[
    [
        'cancel_hour',
        'stock_zone',
        'total_op_cancel',
        'error_count',
        'wrong_pick_count',
        'missing_pick_count',
        'error_rate',
        'hour_avg_error_count',
        'hour_std_error_count',
        'additional_cancel_rate'
    ]
].round(3).head(20))

시간별 이상 로케이션 탐지 결과
이상 로케이션 감지 건수: 25건
           cancel_hour stock_zone  ...  hour_std_error_count  additional_cancel_rate
3  2024-01-01 08:00:00          D  ...                 2.168                   0.417
6  2024-01-01 09:00:00          B  ...                 0.894                   0.562
7  2024-01-01 09:00:00          C  ...                 0.894                   0.364
10 2024-01-01 10:00:00          A  ...                 3.808                   0.536
13 2024-01-01 10:00:00          D  ...                 3.808                   0.355
15 2024-01-01 11:00:00          A  ...                 3.162                   0.333
21 2024-01-01 12:00:00          B  ...                 2.490                   0.400
23 2024-01-01 12:00:00          D  ...                 2.490                   0.294
29 2024-01-01 13:00:00          E  ...                 4.561                   0.294
32 2024-01-01 14:00:00          C  ...                 4.336                   0.375
31 2024-01-01 14:00:00      

In [25]:
# =============================================
# STEP 14-2: 이상 로케이션 교육 스케줄 생성
# - 이상 탐지 시간의 다음 1시간에 개선 효과가 발생한다고 가정
# =============================================

training_schedule = outlier_locations[
    ['cancel_hour', 'stock_zone', 'error_count', 'error_rate']
].copy()

training_schedule['training_hour'] = training_schedule['cancel_hour']
training_schedule['effect_hour'] = training_schedule['cancel_hour'] + pd.Timedelta(hours=1)

training_schedule = training_schedule.rename(columns={
    'cancel_hour': 'detected_hour',
    'error_count': 'detected_error_count',
    'error_rate': 'detected_error_rate'
})

print("=" * 60)
print("1시간 단위 교육/피드백 스케줄")
print("=" * 60)
print(training_schedule.head(20))

1시간 단위 교육/피드백 스케줄
         detected_hour stock_zone  ...       training_hour         effect_hour
3  2024-01-01 08:00:00          D  ... 2024-01-01 08:00:00 2024-01-01 09:00:00
6  2024-01-01 09:00:00          B  ... 2024-01-01 09:00:00 2024-01-01 10:00:00
7  2024-01-01 09:00:00          C  ... 2024-01-01 09:00:00 2024-01-01 10:00:00
10 2024-01-01 10:00:00          A  ... 2024-01-01 10:00:00 2024-01-01 11:00:00
13 2024-01-01 10:00:00          D  ... 2024-01-01 10:00:00 2024-01-01 11:00:00
15 2024-01-01 11:00:00          A  ... 2024-01-01 11:00:00 2024-01-01 12:00:00
21 2024-01-01 12:00:00          B  ... 2024-01-01 12:00:00 2024-01-01 13:00:00
23 2024-01-01 12:00:00          D  ... 2024-01-01 12:00:00 2024-01-01 13:00:00
29 2024-01-01 13:00:00          E  ... 2024-01-01 13:00:00 2024-01-01 14:00:00
32 2024-01-01 14:00:00          C  ... 2024-01-01 14:00:00 2024-01-01 15:00:00
31 2024-01-01 14:00:00          B  ... 2024-01-01 14:00:00 2024-01-01 15:00:00
30 2024-01-01 14:00:00          A 

## **STEP15 - 재교육 효과 시나리오**

In [26]:
# =============================================
# STEP 15: 다음 시간 개선 효과 시뮬레이션
# - 교육 대상 로케이션은 다음 시간 오집·미집이 10% 감소한다고 가정
# - 전후 비교 분모는 기준 운영취소 건수로 고정
# =============================================

TRAINING_EFFECT = 0.10

effect_targets = training_schedule[
    ['effect_hour', 'stock_zone']
].drop_duplicates()

effect_target_df = hourly_df.merge(
    effect_targets,
    left_on=['cancel_hour', 'stock_zone'],
    right_on=['effect_hour', 'stock_zone'],
    how='inner'
).copy()

effect_error_df = effect_target_df[
    effect_target_df['error_flag'] == 1
].copy()

current_total_op_cancel = len(hourly_df)
current_error_count = int(hourly_df['error_flag'].sum())
current_add_cancel_count = int(hourly_df['additional_cancel_flag'].sum())

current_error_rate = current_error_count / current_total_op_cancel
current_add_cancel_rate = current_add_cancel_count / current_total_op_cancel

target_next_hour_error_count = len(effect_error_df)
prevented_error_count = target_next_hour_error_count * TRAINING_EFFECT

target_error_add_cancel_rate = effect_error_df['additional_cancel_flag'].mean()
reduced_add_cancel_count = prevented_error_count * target_error_add_cancel_rate

after_error_count = current_error_count - prevented_error_count
after_add_cancel_count = current_add_cancel_count - reduced_add_cancel_count

# 동일한 운영취소 분모로 개선폭 비교
after_error_rate = after_error_count / current_total_op_cancel
after_add_cancel_rate = after_add_cancel_count / current_total_op_cancel

error_reduction_pp = current_error_rate - after_error_rate
add_cancel_reduction_pp = current_add_cancel_rate - after_add_cancel_rate

hourly_training_effect = pd.DataFrame({
    '지표': [
        '기준 운영취소 건수',
        '오집·미집 건수',
        '운영취소 내 오집·미집 비중',
        '추가취소 건수',
        '운영취소 내 추가취소율'
    ],
    '교육 전': [
        current_total_op_cancel,
        current_error_count,
        current_error_rate,
        current_add_cancel_count,
        current_add_cancel_rate
    ],
    '교육 후 예상': [
        current_total_op_cancel,
        after_error_count,
        after_error_rate,
        after_add_cancel_count,
        after_add_cancel_rate
    ],
    '개선 효과': [
        0,
        prevented_error_count,
        error_reduction_pp,
        reduced_add_cancel_count,
        add_cancel_reduction_pp
    ]
})

print("=" * 60)
print("STEP 15. 시간 단위 교육 후 다음 시간 개선 효과")
print("=" * 60)
print(f"교육/피드백 발생 건수: {len(training_schedule):,}건")
print(f"기준 운영 문제 후보 건수: {current_total_op_cancel:,}건")
print(f"다음 시간 효과 적용 대상 오집·미집 건수: {target_next_hour_error_count:,}건")
print(f"오집·미집 감소 가정: {TRAINING_EFFECT:.0%}")
print(f"예상 예방 오집·미집 건수: {prevented_error_count:,.1f}건")
print(f"예상 추가취소 감소 건수: {reduced_add_cancel_count:,.1f}건")
print(f"운영취소 내 오집·미집 비중 개선폭: {error_reduction_pp * 100:.2f}%p")
print(f"운영취소 내 추가취소율 개선폭: {add_cancel_reduction_pp * 100:.2f}%p")

print("\n[교육 전후 기대효과 - 운영취소 기준]")
print(hourly_training_effect)


STEP 15. 시간 단위 교육 후 다음 시간 개선 효과
교육/피드백 발생 건수: 25건
기준 운영 문제 후보 건수: 2,141건
다음 시간 효과 적용 대상 오집·미집 건수: 485건
오집·미집 감소 가정: 10%
예상 예방 오집·미집 건수: 48.5건
예상 추가취소 감소 건수: 18.6건
운영취소 내 오집·미집 비중 개선폭: 2.27%p
운영취소 내 추가취소율 개선폭: 0.87%p

[교육 전후 기대효과 - 운영취소 기준]
                지표         교육 전      교육 후 예상      개선 효과
0       기준 운영취소 건수  2141.000000  2141.000000   0.000000
1         오집·미집 건수  1710.000000  1661.500000  48.500000
2  운영취소 내 오집·미집 비중     0.798692     0.776039   0.022653
3          추가취소 건수   826.000000   807.400000  18.600000
4     운영취소 내 추가취소율     0.385801     0.377113   0.008688


In [27]:
# =============================================
# STEP 15-1: 교육 효과 민감도 분석
# - 동일한 기준 운영취소 건수를 분모로 사용
# =============================================

effect_scenarios = [0.10, 0.20, 0.30]
scenario_results = []

for effect in effect_scenarios:
    prevented_errors = target_next_hour_error_count * effect
    reduced_add_cancels = prevented_errors * target_error_add_cancel_rate

    new_error_count = current_error_count - prevented_errors
    new_add_cancel_count = current_add_cancel_count - reduced_add_cancels

    new_error_rate = new_error_count / current_total_op_cancel
    new_add_cancel_rate = new_add_cancel_count / current_total_op_cancel

    scenario_results.append({
        '다음 시간 오집·미집 감소 가정': f'{effect:.0%}',
        '예상 예방 오집·미집 건수': prevented_errors,
        '예상 추가취소 감소 건수': reduced_add_cancels,
        '교육 후 운영취소 내 오집·미집 비중': new_error_rate,
        '오집·미집 비중 개선폭(%p)': (current_error_rate - new_error_rate) * 100,
        '교육 후 운영취소 내 추가취소율': new_add_cancel_rate,
        '추가취소율 개선폭(%p)': (current_add_cancel_rate - new_add_cancel_rate) * 100
    })

hourly_scenario_df = pd.DataFrame(scenario_results)

print("=" * 60)
print("시간 단위 교육 효과 민감도 분석")
print("=" * 60)
print(hourly_scenario_df.round(4))


시간 단위 교육 효과 민감도 분석
  다음 시간 오집·미집 감소 가정  예상 예방 오집·미집 건수  ...  교육 후 운영취소 내 추가취소율  추가취소율 개선폭(%p)
0               10%            48.5  ...             0.3771         0.8688
1               20%            97.0  ...             0.3684         1.7375
2               30%           145.5  ...             0.3597         2.6063

[3 rows x 7 columns]


In [28]:
# =============================================
# STEP 15-2: 시간별 교육 효과 추적 테이블
# =============================================

# 효과 대상 시간 × 로케이션별 실제 다음 시간 오집·미집 집계
effect_hour_summary = (
    effect_target_df
    .groupby(['cancel_hour', 'stock_zone'])
    .agg(
        next_hour_total_op_cancel=('order_id', 'count'),
        next_hour_error_count=('error_flag', 'sum'),
        next_hour_add_cancel_count=('additional_cancel_flag', 'sum')
    )
    .reset_index()
)

effect_hour_summary['expected_prevented_error_count'] = (
    effect_hour_summary['next_hour_error_count'] * TRAINING_EFFECT
)

effect_hour_summary['expected_reduced_add_cancel_count'] = (
    effect_hour_summary['expected_prevented_error_count'] *
    target_error_add_cancel_rate
)

effect_hour_summary = effect_hour_summary.sort_values(
    ['cancel_hour', 'expected_prevented_error_count'],
    ascending=[True, False]
)

print("=" * 60)
print("시간별/로케이션별 다음 시간 개선 효과 추정")
print("=" * 60)

print(effect_hour_summary.round(3).head(30))

시간별/로케이션별 다음 시간 개선 효과 추정
           cancel_hour  ... expected_reduced_add_cancel_count
0  2024-01-01 09:00:00  ...                             0.307
1  2024-01-01 10:00:00  ...                             0.652
2  2024-01-01 10:00:00  ...                             0.614
4  2024-01-01 11:00:00  ...                             1.035
3  2024-01-01 11:00:00  ...                             0.997
5  2024-01-01 12:00:00  ...                             1.189
7  2024-01-01 13:00:00  ...                             1.227
6  2024-01-01 13:00:00  ...                             1.112
8  2024-01-01 14:00:00  ...                             0.767
9  2024-01-01 15:00:00  ...                             1.112
10 2024-01-01 15:00:00  ...                             0.805
11 2024-01-01 15:00:00  ...                             0.614
12 2024-01-01 16:00:00  ...                             0.767
13 2024-01-01 17:00:00  ...                             1.074
14 2024-01-01 18:00:00  ...                  

In [29]:
# =============================================
# STEP 15-3: 시각화
# =============================================

import matplotlib.pyplot as plt

# 시간대별 이상 로케이션 수
outlier_by_hour = (
    outlier_locations
    .groupby('cancel_hour')
    .size()
    .reset_index(name='outlier_location_count')
)

# 시간대별 기대 예방 건수
effect_by_hour = (
    effect_hour_summary
    .groupby('cancel_hour')
    .agg(
        expected_prevented_error_count=('expected_prevented_error_count', 'sum'),
        expected_reduced_add_cancel_count=('expected_reduced_add_cancel_count', 'sum')
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(
    outlier_by_hour['cancel_hour'],
    outlier_by_hour['outlier_location_count'],
    marker='o'
)
axes[0].set_title('시간대별 이상 로케이션 감지 수')
axes[0].set_xlabel('시간')
axes[0].set_ylabel('이상 로케이션 수')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(
    effect_by_hour['cancel_hour'],
    effect_by_hour['expected_prevented_error_count'],
    marker='o',
    label='오집·미집 예방'
)
axes[1].plot(
    effect_by_hour['cancel_hour'],
    effect_by_hour['expected_reduced_add_cancel_count'],
    marker='o',
    label='추가취소 감소'
)
axes[1].set_title('시간대별 다음 시간 개선 효과 추정')
axes[1].set_xlabel('효과 적용 시간')
axes[1].set_ylabel('건수')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

In [30]:
# =============================================
# STEP 15-4: 자동화 후 추가 기능 효과 및 비용 절감 요약
# =============================================

AVG_COST_PER_ADD_CANCEL = 3000  # 실제 비용 아님: 임의 가정 단가(원/건)

daily_prevented_error_count = prevented_error_count
daily_reduced_add_cancel_count = reduced_add_cancel_count
daily_expected_saving = daily_reduced_add_cancel_count * AVG_COST_PER_ADD_CANCEL

daily_error_reduction_pp = daily_prevented_error_count / current_total_op_cancel
daily_add_cancel_reduction_pp = daily_reduced_add_cancel_count / current_total_op_cancel

print(
    f"자동화 개선 후 남아 있는 운영취소를 기준으로, "
    f"1시간 단위 이상 로케이션 교육 기능을 추가 적용하면 "
    f"하루 오집·미집 약 {daily_prevented_error_count:.1f}건을 예방하고, "
    f"추가취소 약 {daily_reduced_add_cancel_count:.1f}건을 추가로 줄일 수 있을 것으로 추정된다. "
    f"이는 기준 운영취소 대비 오집·미집 비중을 "
    f"{daily_error_reduction_pp * 100:.2f}%p, "
    f"추가취소율을 {daily_add_cancel_reduction_pp * 100:.2f}%p 추가 개선하는 효과에 해당한다. "
    f"건당 평균 손실비용 {AVG_COST_PER_ADD_CANCEL:,}원 적용 시 "
    f"하루 약 {daily_expected_saving:,.0f}원의 추가 비용 절감 잠재효과가 있다."
)


자동화 개선 후 남아 있는 운영취소를 기준으로, 1시간 단위 이상 로케이션 교육 기능을 추가 적용하면 하루 오집·미집 약 48.5건을 예방하고, 추가취소 약 18.6건을 추가로 줄일 수 있을 것으로 추정된다. 이는 기준 운영취소 대비 오집·미집 비중을 2.27%p, 추가취소율을 0.87%p 추가 개선하는 효과에 해당한다. 건당 평균 손실비용 3,000원 적용 시 하루 약 55,800원의 추가 비용 절감 잠재효과가 있다.


In [31]:
# =============================================
# STEP 16: 기능 추가 시 월별 / 연간 추가 개선 효과 추정
# - 건수 효과 + 운영취소 내 개선폭을 함께 추정
# =============================================

base_daily_prevented_error = daily_prevented_error_count
base_daily_reduced_add_cancel = daily_reduced_add_cancel_count
base_daily_op_cancel = current_total_op_cancel
base_daily_orders = N_ORDERS

days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

quarterly_daily_order_increase = 20000
monthly_daily_growth = quarterly_daily_order_increase / 3
ORDER_VOLUME_ERROR_RATE = 0.05
AVG_COST_PER_ADD_CANCEL = 3000  # 실제 비용 아님: 임의 가정 단가(원/건)

np.random.seed(42)
monthly_feature_results = []

for month in range(1, 13):
    daily_orders = base_daily_orders + ((month - 1) * monthly_daily_growth)

    order_volume_multiplier = np.random.uniform(
        1 - ORDER_VOLUME_ERROR_RATE,
        1 + ORDER_VOLUME_ERROR_RATE
    )

    daily_orders_adj = int(daily_orders * order_volume_multiplier)
    order_scale = daily_orders_adj / base_daily_orders

    monthly_op_cancel = (
        base_daily_op_cancel * order_scale * days_in_month[month - 1]
    )

    monthly_prevented_error = (
        base_daily_prevented_error * order_scale * days_in_month[month - 1]
    )

    monthly_reduced_add_cancel = (
        base_daily_reduced_add_cancel * order_scale * days_in_month[month - 1]
    )

    monthly_cost_saving = monthly_reduced_add_cancel * AVG_COST_PER_ADD_CANCEL

    monthly_error_reduction_rate = (
        monthly_prevented_error / monthly_op_cancel
        if monthly_op_cancel > 0 else 0
    )
    monthly_add_cancel_reduction_rate = (
        monthly_reduced_add_cancel / monthly_op_cancel
        if monthly_op_cancel > 0 else 0
    )

    monthly_feature_results.append({
        'month': month,
        'days': days_in_month[month - 1],
        'daily_orders': daily_orders_adj,
        'order_scale': order_scale,
        'monthly_op_cancel_count': monthly_op_cancel,
        'monthly_prevented_error_count': monthly_prevented_error,
        'monthly_error_reduction_rate': monthly_error_reduction_rate,
        'monthly_reduced_add_cancel_count': monthly_reduced_add_cancel,
        'monthly_add_cancel_reduction_rate': monthly_add_cancel_reduction_rate,
        'monthly_cost_saving': monthly_cost_saving
    })

monthly_feature_effect_df = pd.DataFrame(monthly_feature_results)

annual_op_cancel = monthly_feature_effect_df['monthly_op_cancel_count'].sum()
annual_prevented_error = monthly_feature_effect_df['monthly_prevented_error_count'].sum()
annual_reduced_add_cancel = monthly_feature_effect_df['monthly_reduced_add_cancel_count'].sum()
annual_cost_saving = monthly_feature_effect_df['monthly_cost_saving'].sum()

annual_error_reduction_rate = annual_prevented_error / annual_op_cancel
annual_add_cancel_reduction_rate = annual_reduced_add_cancel / annual_op_cancel

print("=" * 60)
print("STEP 16. 기능 추가 시 월별 / 연간 추가 개선 효과 추정")
print("=" * 60)

print("\n[월별 추가 개선 효과]")
print(
    monthly_feature_effect_df[
        [
            'month',
            'days',
            'daily_orders',
            'monthly_op_cancel_count',
            'monthly_prevented_error_count',
            'monthly_error_reduction_rate',
            'monthly_reduced_add_cancel_count',
            'monthly_add_cancel_reduction_rate',
            'monthly_cost_saving'
        ]
    ].round(4).to_string(index=False)
)

print("\n[연간 추가 개선 효과 요약]")
print(f"연간 예상 운영취소 건수: {annual_op_cancel:,.0f}건")
print(f"연간 오집·미집 예방 건수: {annual_prevented_error:,.0f}건")
print(f"운영취소 내 오집·미집 비중 개선폭: {annual_error_reduction_rate * 100:.2f}%p")
print(f"연간 추가취소 감소 건수: {annual_reduced_add_cancel:,.0f}건")
print(f"운영취소 내 추가취소율 개선폭: {annual_add_cancel_reduction_rate * 100:.2f}%p")
print(f"연간 추가 비용 절감 기대액: {annual_cost_saving:,.0f}원")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    monthly_feature_effect_df['month'],
    monthly_feature_effect_df['monthly_reduced_add_cancel_count'],
    marker='o',
    color='#55A868'
)
axes[0].set_title('월별 추가취소 감소 예상 건수')
axes[0].set_xlabel('월')
axes[0].set_ylabel('추가취소 감소 건수')
axes[0].set_xticks(range(1, 13))

axes[1].plot(
    monthly_feature_effect_df['month'],
    monthly_feature_effect_df['monthly_add_cancel_reduction_rate'] * 100,
    marker='o',
    color='#4C72B0'
)
axes[1].set_title('운영취소 내 추가취소율 개선폭')
axes[1].set_xlabel('월')
axes[1].set_ylabel('개선폭 (%p)')
axes[1].set_xticks(range(1, 13))

plt.tight_layout()
plt.show()


STEP 16. 기능 추가 시 월별 / 연간 추가 개선 효과 추정

[월별 추가 개선 효과]
 month  days  daily_orders  monthly_op_cancel_count  monthly_prevented_error_count  monthly_error_reduction_rate  monthly_reduced_add_cancel_count  monthly_add_cancel_reduction_rate  monthly_cost_saving
     1    31        148118               65538.2652                      1484.6361                        0.0227                          569.3656                             0.0087          1708096.776
     2    28        163727               65434.0413                      1482.2751                        0.0227                          568.4601                             0.0087          1705380.432
     3    31        167122               73947.0284                      1675.1195                        0.0227                          642.4170                             0.0087          1927250.904
     4    30        171677               73512.0914                      1665.2669                        0.0227                        

In [32]:
# =============================================
# STEP 16-2: 연간 추가 개선 효과 한 줄 요약
# =============================================

annual_prevented_error = monthly_feature_effect_df['monthly_prevented_error_count'].sum()
annual_reduced_add_cancel = monthly_feature_effect_df['monthly_reduced_add_cancel_count'].sum()
annual_cost_saving = monthly_feature_effect_df['monthly_cost_saving'].sum()
annual_op_cancel = monthly_feature_effect_df['monthly_op_cancel_count'].sum()

annual_error_reduction_rate = annual_prevented_error / annual_op_cancel
annual_add_cancel_reduction_rate = annual_reduced_add_cancel / annual_op_cancel

print(
    f"자동화 이후 1시간 단위 이상 로케이션 교육 기능을 추가 적용하면, "
    f"연간 오집·미집 약 {annual_prevented_error:,.0f}건을 예방하고 "
    f"추가취소 약 {annual_reduced_add_cancel:,.0f}건을 추가로 줄일 수 있을 것으로 추정된다. "
    f"이는 연간 예상 운영취소 대비 오집·미집 비중 "
    f"{annual_error_reduction_rate * 100:.2f}%p, "
    f"추가취소율 {annual_add_cancel_reduction_rate * 100:.2f}%p의 추가 개선 효과에 해당한다. "
    f"건당 평균 손실비용 {AVG_COST_PER_ADD_CANCEL:,}원 기준 "
    f"연간 약 {annual_cost_saving/10000:,.0f}만원의 추가 비용 절감 잠재효과가 있다."
)


자동화 이후 1시간 단위 이상 로케이션 교육 기능을 추가 적용하면, 연간 오집·미집 약 22,084건을 예방하고 추가취소 약 8,469건을 추가로 줄일 수 있을 것으로 추정된다. 이는 연간 예상 운영취소 대비 오집·미집 비중 2.27%p, 추가취소율 0.87%p의 추가 개선 효과에 해당한다. 건당 평균 손실비용 3,000원 기준 연간 약 2,541만원의 추가 비용 절감 잠재효과가 있다.


### 확장 분석 STEP 14~16의 해석 범위
가상 로케이션에서 타토트 오집·부족집품 **수요**의 집중 구간을 탐색하는 아이디어 시연. 과집 공급 사건은 이 교육 분석에서 제외하며, 교육에 따른 공급원 변화·재매칭을 모형화하지 않는다. 전체 집품 분모의 실측 오류율이 아니며 교육효과 10~30%, 취소 1건당 3,000원은 비실측 가정이다. 메인 KPI 및 감가와 합산 금지.

### 이전 버전 수치 사용 주의
v10은 과집 공급과 타토트 오집 수요를 구분하지 않아 2,254건 모두를 대체/복구 수요로 계산했습니다. v12 이후에는 과집을 공급 전용으로 처리하므로 기존 예방률 및 대체율 분모와 다릅니다.

### 추가 기능의 금액 추정 주의

STEP 15~16의 재교육·이상 구역 후속 분석은 **임의 교육효과 및 추가취소 단가 시나리오**로, 메인 운영 KPI 또는 재진열 지연 감가 시나리오의 금액에 합산하지 않습니다. 과거 고정된 금액/개선폭은 최종 수치로 사용하지 않습니다.


### 과거 버전 변경 이력 — 비활성 참고 기록

이전 시간창 및 경제효과 변경 기록은 현재 최종 실행 기준과 다릅니다. 최종 수치·기준은 STEP 5, STEP 11-D/E, STEP 12 및 README를 참조하세요.


### 과거 버전 변경 이력 — 비활성 참고 기록

이전 시간창 및 경제효과 변경 기록은 현재 최종 실행 기준과 다릅니다. 최종 수치·기준은 STEP 5, STEP 11-D/E, STEP 12 및 README를 참조하세요.


### 과거 버전 변경 이력 — 비활성 참고 기록

이전 시간창 및 경제효과 변경 기록은 현재 최종 실행 기준과 다릅니다. 최종 수치·기준은 STEP 5, STEP 11-D/E, STEP 12 및 README를 참조하세요.


### 과거 버전 변경 이력 — 비활성 참고 기록

이전 시간창 및 경제효과 변경 기록은 현재 최종 실행 기준과 다릅니다. 최종 수치·기준은 STEP 5, STEP 11-D/E, STEP 12 및 README를 참조하세요.


In [33]:
# =============================================
# STEP 11-F (기존 검증): 토트·실물회수·정상 SKU 대체·운영취소 정합성
# =============================================
assert orders_df['unit_id'].is_unique
assert orders_df['order_id'].is_unique
assert orders_df['tote_unit_no'].ge(1).all()
assert orders_df['tote_units_total'].between(TOTE_CAPACITY_MIN,TOTE_CAPACITY_MAX).all()
tote_sizes=orders_df.groupby('tote_id').size()
assert tote_sizes.between(TOTE_CAPACITY_MIN,TOTE_CAPACITY_MAX).all()
assert (orders_df['tote_id'].map(tote_sizes)==orders_df['tote_units_total']).all()
assert set(base_cancel_df['order_id'])==set(before['order_id'])|set(base_cancel_df.loc[~base_cancel_df['cancel_reason'].isin(OP_REASONS),'order_id'])
for label,frame in [('수동',before),('자동화',after)]:
    op=frame[frame['cancel_reason'].isin(OP_REASONS)]
    assert frame['order_id'].is_unique and frame['unit_id'].is_unique
    assert (op['tote_rescue_flag']+op['reused_flag']+op['op_cancel_confirmed_flag']).eq(1).all()
    assert op['op_cancel_prevented_flag'].eq(op['tote_rescue_flag']+op['reused_flag']).all()
    assert op['additional_cancel_flag'].eq(op['op_cancel_confirmed_flag']).all()
    assert (frame.loc[frame['tote_rescue_flag'].eq(1),'cancel_reason']=='토트분실').all()
    assert frame.loc[frame['tote_rescue_flag'].eq(1),'source_assignment_released_flag'].eq(0).all()
    assert frame.loc[frame['tote_rescue_flag'].eq(1),'restock_flag'].eq(0).all()
    assert frame.loc[frame['tote_rescue_flag'].eq(1),'consumed_candidate_order_id'].isna().all()
    assert frame.loc[frame['tote_rescue_flag'].eq(1),'process_time'].isna().all()
    assert frame.loc[frame['tote_rescue_flag'].eq(1),'tote_search_detect_time'].le(
        frame.loc[frame['tote_rescue_flag'].eq(1),'tote_cancel_deadline']).all()
    assert frame.loc[frame['reused_flag'].eq(1),'consumed_candidate_order_id'].notna().all()
    assert frame.loc[frame['reused_flag'].eq(0),'consumed_candidate_order_id'].isna().all()
    assert frame['consumed_candidate_order_id'].dropna().is_unique
    assert frame.loc[frame['reused_flag'].eq(1),'matched_candidate_available_time'].le(
        frame.loc[frame['reused_flag'].eq(1),'reuse_ready_time']).all()
    assert frame.loc[frame['reused_flag'].eq(1),'consumed_candidate_order_id'].ne(
        frame.loc[frame['reused_flag'].eq(1),'order_id']).all()
    assert frame.loc[frame['cancel_reason'].isin(['미집','집품파손','리빈파손']),'physical_inspection_pass_flag'].eq(0).all()
    assert frame.loc[frame['cancel_reason'].isin(OP_REASONS)&frame['tote_rescue_flag'].eq(0),'search_time_min'].notna().all()
    assert frame.loc[frame['cancel_reason'].eq('토트분실')&frame['tote_rescue_flag'].eq(0),'process_time'].ge(
        frame.loc[frame['cancel_reason'].eq('토트분실')&frame['tote_rescue_flag'].eq(0),'tote_cancel_deadline']).all()
    # 실물 취소존 도착 전에는 재고처리·원토트 탐색이 불가하다.
    assert frame.loc[frame['cancel_zone_processed_at'].notna(),'cancel_zone_processed_at'].ge(
        frame.loc[frame['cancel_zone_processed_at'].notna(),'cancel_zone_arrived_at']).all()
    physical=frame[frame['cancel_zone_processed_at'].notna()]
    assert physical['cancel_zone_queue_started_at'].ge(physical['cancel_zone_arrived_at']).all()
    assert physical['cancel_zone_processed_at'].ge(physical['cancel_zone_queue_started_at']).all()
    assert physical['cancel_zone_queue_wait_min'].ge(0).all()
    assert np.allclose((physical['cancel_zone_queue_started_at']-physical['cancel_zone_arrived_at']).dt.total_seconds()/60,
                       physical['cancel_zone_queue_wait_min'],atol=1e-3)
    expected=frame['expected_line_arrived_at']-frame['pick_completed_at']
    assert expected.dt.total_seconds().div(60).between(*LINE_TRANSIT_MIN).all()
    ordinary=frame[frame['cancel_reason'].isin(['집품파손','리빈파손'])]
    assert (ordinary['cancel_zone_arrived_at']-ordinary['line_arrived_at']).dt.total_seconds().div(60).between(*CANCEL_ZONE_TRANSIT_MIN).all()
    assert ordinary['queue_entry_time'].eq(ordinary['cancel_zone_arrived_at']).all()
    assert ordinary['process_time'].ge(ordinary['queue_entry_time']).all()
    absent=frame[frame['cancel_reason'].isin(['오집','미집'])]
    assert absent['physical_unit_id'].isna().all()
    assert absent['cancel_zone_arrived_at'].isna().all() and absent['cancel_zone_processed_at'].isna().all()
    assert absent['queue_entry_time'].eq(absent['demand_request_at']).all()
    assert absent['process_time'].ge(absent['demand_request_at']).all()
    assert absent['physical_fifo_service_count'].eq(0).all()
    tt=frame[frame['cancel_reason'].eq('토트분실')]
    assert tt.loc[tt['tote_rescue_flag'].eq(1),'tote_search_detect_time'].le(
        tt.loc[tt['tote_rescue_flag'].eq(1),'tote_cancel_deadline']).all()
    # 기존 경로 조회로 라인에서 직접 처리한 사건은 취소존을 통과하지 않는다.
    existing=tt[tt['tote_existing_route_rescued_flag'].eq(1)]
    assert existing['tote_rescue_flag'].eq(1).all()
    assert existing['tote_cancel_zone_required_flag'].eq(0).all()
    assert existing['cancel_zone_arrived_at'].isna().all()
    assert existing['cancel_zone_processed_at'].isna().all()
    assert existing['queue_entry_time'].isna().all()
    assert existing['tote_search_detect_time'].le(existing['tote_cancel_deadline']).all()
    zone=tt[tt['tote_cancel_zone_required_flag'].eq(1)]
    assert zone.loc[zone['tote_rescue_flag'].eq(1),'cancel_zone_processed_at'].ge(
        zone.loc[zone['tote_rescue_flag'].eq(1),'cancel_zone_arrived_at']).all()
    assert zone['tote_existing_route_rescued_flag'].eq(0).all()
    assert tt['tote_existing_route_traced_flag'].equals(tt['order_id'].map(
        lambda oid:int(stable_uniform_local(oid,'legacy_tote_route_trace')<EXISTING_TOTE_ROUTE_TRACE_PROB)))
    assert tt.loc[tt['tote_rescue_flag'].eq(0),'process_time'].ge(
        tt.loc[tt['tote_rescue_flag'].eq(0),'tote_cancel_deadline']).all()
    assert tt.loc[tt['tote_manual_search_found_flag'].eq(0),'tote_rescue_flag'].eq(0).all()
    routed_zone=tt[tt['tote_recovery_class'].eq('within_30') & tt['tote_cancel_zone_required_flag'].eq(1)]
    assert np.allclose((routed_zone['cancel_zone_arrived_at']-
        routed_zone['line_arrived_at']).dt.total_seconds()/60, LOST_TOTE_TO_ZONE_MIN)
    t=frame[frame['cancel_reason'].eq('토트분실')]
    assert t['tote_return_time'].notna().all()  # 수일 늦게라도 전원 회수되며 물리적 영구분실 아님.
    assert set(t['tote_recovery_class'])<= {'within_30','late_same_day','multi_day'}
    assert t['tote_return_within_30_flag'].eq(t['tote_recovery_class'].eq('within_30')).all()
    assert t['tote_delayed_return_flag'].eq(t['tote_recovery_class'].eq('late_same_day')).all()
    assert t['tote_multi_day_recovery_flag'].eq(t['tote_recovery_class'].eq('multi_day')).all()
    late=t[t['tote_recovery_class'].eq('late_same_day')]
    rare=t[t['tote_recovery_class'].eq('multi_day')]
    assert late['tote_rescue_flag'].eq(0).all() and rare['tote_rescue_flag'].eq(0).all()
    assert late['tote_return_time'].gt(late['process_time']).all()
    assert rare['tote_return_time'].gt(rare['process_time']+pd.Timedelta(days=1)).all()
    assert rare['tote_return_time'].map(operating_day).ne(rare['cancel_time'].map(operating_day)).all()
    assert len(op)==op['op_cancel_prevented_flag'].sum()+op['op_cancel_confirmed_flag'].sum()
assert set(before['order_id'])==set(after['order_id'])
ma=before.set_index('order_id');aa=after.set_index('order_id')
for col in ('cancel_time','pick_completed_at','expected_line_arrived_at','problem_detected_at','cancel_reason','tote_id','tote_unit_no','tote_units_total',
            'tote_return_within_30_flag','tote_recovery_class','tote_multi_day_recovery_flag','tote_existing_route_traced_flag','tote_existing_route_rescued_flag',
            'customer_cancel_return_time','physical_inspection_pass_flag','tote_cancel_deadline'):
    assert ma[col].sort_index().equals(aa[col].sort_index()),(col,'수동/자동 동일 물리 사건 검증 오류')
within_ids=ma.index[(ma['cancel_reason']=='토트분실')&(ma['tote_return_within_30_flag']==1)]
assert ma.loc[within_ids,'tote_return_time'].equals(aa.loc[within_ids,'tote_return_time'])
for frame in (ma,aa):
    late=frame[(frame['cancel_reason']=='토트분실')&(frame['tote_recovery_class']=='late_same_day')]
    lo,hi=TOTE_LATE_SAME_DAY_RETURN_DELAY_RANGE
    lag=late.index.to_series().map(lambda oid:lo+(hi-lo)*stable_uniform_local(oid,'forced_cancel_return_lag'))
    observed=(late['tote_return_time']-late['process_time']).dt.total_seconds()/60
    assert np.allclose(observed,lag.to_numpy(),atol=1e-5)
assert int(before['op_cancel_confirmed_flag'].sum())==manual_confirmed
assert int(after['op_cancel_confirmed_flag'].sum())==auto_confirmed
# 시나리오별 실제 탐색 실수는 물리적 수동 탐색에만 적용.
for frame in (ma,aa):
    tt=frame[(frame['cancel_reason']=='토트분실')&(frame['tote_recovery_class']=='within_30')]
    assert (tt['tote_search_detect_time'].notna()).all()
    for label,part in [('정상 실물 오류',frame[frame['cancel_reason'].isin(['집품파손','리빈파손'])]),
                       ('토트-취소존 경유',tt[tt['tote_cancel_zone_required_flag'].eq(1)])]:
        assert part['cancel_zone_arrived_at'].ge(part['line_arrived_at']).all(),label
print('PASS: 기존 WMS 경로조회 80% 공통·기한 내 복귀시 취소존 미진입, FIFO 인과관계, 20·30분 한도, 물리탐색 10% 실수, 90·9·1% 회수, 중복 재고 소비 없음')


# v11 신규 무결성: 과집 공급 전용, 타토트 오집 수요-공급 1:1, 품목 동일.
for whole in (final_df,match_input_df):
    only=whole.loc[whole['pick_supply_only_flag'].eq(1)]
    assert only['demand_flag'].eq(0).all()
    assert only[['reused_flag','op_cancel_confirmed_flag','op_cancel_prevented_flag','tool_used_flag']].eq(0).all().all()
    assert only['queue_processing_started_at'].ge(only['cancel_zone_arrived_at']).all()
    assert only['process_time'].ge(only['queue_processing_started_at']).all()
    assert only['qty_diff'].eq(1).all()
    assert only['physical_unit_id'].is_unique
    wrong=whole.loc[whole['cancel_reason'].eq('오집')].set_index('error_event_id')
    supply=whole.loc[whole['cancel_reason'].eq('타토트오집_공급')].set_index('error_event_id')
    assert wrong.index.is_unique and supply.index.is_unique and set(wrong.index)==set(supply.index)
    for col in ['product_id','sku_id','expected_tote_id','actual_tote_id','planned_unit_id']:
        assert wrong[col].sort_index().equals(supply[col].sort_index()),col
    assert wrong['qty_diff'].eq(-1).all() and supply['qty_diff'].eq(1).all()
    assert wrong['expected_tote_id'].ne(wrong['actual_tote_id']).all()
    assert supply['tote_id'].eq(supply['actual_tote_id']).all()
    assert wrong['restock_flag'].eq(0).all() and wrong['physical_inspection_pass_flag'].eq(0).all()
    assert whole.loc[whole['cancel_reason'].eq('미집'),'qty_diff'].eq(-1).all()
    consumed=whole.loc[whole['reused_flag'].eq(1)].copy()
    check=consumed.merge(whole[['order_id','product_id','sku_id','inspection_flag','source_assignment_released_flag']],left_on='consumed_candidate_order_id',right_on='order_id',suffixes=('_target','_source'),validate='one_to_one')
    assert len(check)==len(consumed)
    assert check['product_id_target'].eq(check['product_id_source']).all()
    assert check['sku_id_target'].eq(check['sku_id_source']).all()
    assert check['inspection_flag_source'].eq(1).all() and check['source_assignment_released_flag_source'].eq(1).all()
    assert len(whole.loc[whole['cancel_reason'].isin(['단순취소','고객취소_집품후'])])==len(original_cancel_df.loc[original_cancel_df['cancel_reason'].isin(['단순취소','고객취소_집품후'])])
    assert len(whole.loc[whole['demand_flag'].eq(1)]) + len(whole.loc[whole['cancel_reason'].eq('과집')])==int(original_cancel_df['cancel_reason'].isin(OP_REASONS).sum())
print('PASS: v12 과집 공급 전용, 타토트 오집 수요-공급 연결/수량 보존, 부족집품 공급 제외, 공급 실물 중복소비 금지')

# 최종 요약: 공급 활용률은 수요 기준 대체율과 별도.
for label,whole in [('수동',final_df),('자동화',match_input_df)]:
    op=whole[whole['demand_flag'].eq(1)]
    eligible=whole[whole['cancel_reason'].isin(SUPPLY_REASONS)&whole['inspection_flag'].eq(1)&whole['source_assignment_released_flag'].eq(1)].copy()
    available=eligible[['process_time','cancel_zone_processed_at']].max(axis=1)
    tm=eligible['cancel_reason'].eq('토트분실')
    available.loc[tm]=eligible.loc[tm,['tote_return_time','process_time','cancel_zone_processed_at','queue_entry_time']].max(axis=1)
    eligible=eligible[available.map(operating_day).eq(eligible['cancel_time'].map(operating_day)) & available.map(operating_day).notna()]
    linked=op.loc[op['cancel_reason'].eq('오집')&op['reused_flag'].eq(1)]
    own_linked=int(linked['consumed_candidate_order_id'].eq(linked['error_event_id']+'-SUP').sum())
    print(label,'당일 가용 공급원',len(eligible),'대체',int(op['reused_flag'].sum()),'자기 오집 회수 실물로 대체',own_linked)
    print(whole.groupby('cancel_reason').agg(행수=('order_id','size'),등록검수=('inspection_flag','sum'),대체=('reused_flag','sum'),원토트복구=('tote_rescue_flag','sum'),확정=('op_cancel_confirmed_flag','sum')).to_string())

# 1실물 1회, 무실물 0회 / 토트별 시간 및 FIFO 검증
for label,whole in [('수동',final_df),('자동',match_input_df)]:
    assert whole['virtual_fifo_service_count'].eq(0).all()
    assert whole['physical_fifo_service_count'].between(0,1).all()
    wrong=whole[whole['picking_error_type'].eq('타토트오집')]
    assert wrong.groupby('error_event_id')['physical_fifo_service_count'].sum().eq(1).all()
    assert whole.loc[whole['cancel_reason'].eq('미집'),'physical_fifo_service_count'].eq(0).all()
    assert whole.loc[whole['cancel_reason'].eq('과집'),'physical_fifo_service_count'].eq(1).all()
    ordinary=whole[whole['cancel_reason'].isin(['과집','타토트오집_공급','집품파손','리빈파손'])]
    for col in ['pick_completed_at','line_arrived_at','cancel_zone_arrived_at']:
        assert ordinary.groupby('tote_id')[col].nunique().le(1).all(),col
    paired=wrong.groupby('error_event_id')['pick_completed_at'].agg(['min','max'])
    assert (paired['max']-paired['min']).dt.total_seconds().le(TOTE_COMPLETION_SPREAD_MIN*60).all()
    assert paired['max'].gt(paired['min']).any()
    assert whole['pick_completed_at'].eq(whole['tote_id'].map(tote_pick_completed)).all()
    physical=whole[whole['physical_fifo_service_count'].eq(1)].sort_values(['cancel_zone_queue_started_at','order_id'])
    assert physical['cancel_zone_processed_at'].notna().all()
    assert (physical['cancel_zone_queue_started_at'].iloc[1:].to_numpy()>=physical['cancel_zone_processed_at'].iloc[:-1].to_numpy()).all()
    assert physical['cancel_zone_arrived_at'].is_monotonic_increasing
    print(label,'실물 FIFO 처리수',len(physical),'평균 실물 큐 대기분',physical['cancel_zone_queue_wait_min'].mean())
print('PASS: 오집 수요·공급 1회 실물 처리, 미집 0회, 토트 시간 및 FIFO')

# 토트별 시간 검증: 동시 완료 허용, 순차 완료 강제 없음.
assert all(tote_pick_completed[t]>=tote_times[t] for t in tote_ids)
_pair_gaps=pd.Series([abs((tote_pick_completed[t]-tote_pick_completed[neighbor[t]]).total_seconds())/60 for t in tote_ids])
assert _pair_gaps.le(TOTE_COMPLETION_SPREAD_MIN).all() and _pair_gaps.gt(0).any()
_pair_direction=pd.Series([(tote_pick_completed[t]-tote_pick_completed[neighbor[t]]).total_seconds() for t in tote_ids if t<neighbor[t]])
assert _pair_direction.gt(0).any() and _pair_direction.lt(0).any()
print('연관 토트 완료시각 차이(분): 평균',_pair_gaps.mean(),'최대',_pair_gaps.max(),'동시 완료 비중',_pair_gaps.eq(0).mean())

# v14 — 고객취소 FIFO 검증
customer_ids=[]
for label,whole in [('수동',final_df),('자동',match_input_df)]:
    post=whole[whole['cancel_reason'].eq('고객취소_집품후')]
    pre=whole[whole['cancel_reason'].eq('단순취소')]
    assert len(post)==round((len(post)+len(pre))*POST_PICK_CUSTOMER_CANCEL_RATIO)
    assert post['physical_fifo_service_count'].eq(1).all()
    assert pre['physical_fifo_service_count'].eq(0).all() and pre['cancel_zone_arrived_at'].isna().all()
    assert post['queue_entry_time'].eq(post['customer_cancel_return_time']).all()
    assert post['process_time'].eq(post['cancel_zone_processed_at']).all()
    assert post['cancel_zone_queue_started_at'].ge(post['customer_cancel_return_time']).all()
    assert post['demand_flag'].eq(0).all() and post['op_cancel_confirmed_flag'].eq(0).all()
    used=whole[whole['consumed_candidate_order_id'].isin(post['order_id'])].merge(post[['order_id','cancel_zone_processed_at','inspection_flag']],left_on='consumed_candidate_order_id',right_on='order_id',suffixes=('','_source'),validate='one_to_one')
    assert used['matched_candidate_available_time'].eq(used['cancel_zone_processed_at_source']).all()
    assert used['reuse_ready_time'].ge(used['cancel_zone_processed_at_source']).all()
    assert used['inspection_flag_source'].eq(1).all()
    customer_ids.append(set(post['order_id']))
    print(label,'집품 전/후',len(pre),len(post),'검수통과',post['inspection_flag'].sum(),'고객취소 큐 대기',post['cancel_zone_queue_wait_min'].mean(),'고객취소 공급 사용',len(used))
    print(used.groupby('cancel_reason').size().to_string())
assert customer_ids[0]==customer_ids[1]
print('PASS: 고객취소 FIFO 1회, 처리완료 후 공급, 운영취소 분모 제외')


PASS: 기존 WMS 경로조회 80% 공통·기한 내 복귀시 취소존 미진입, FIFO 인과관계, 20·30분 한도, 물리탐색 10% 실수, 90·9·1% 회수, 중복 재고 소비 없음
PASS: v12 과집 공급 전용, 타토트 오집 수요-공급 연결/수량 보존, 부족집품 공급 제외, 공급 실물 중복소비 금지
수동 당일 가용 공급원 1605 대체 1026 자기 오집 회수 실물로 대체 681
                 행수  등록검수   대체  원토트복구   확정
cancel_reason                             
고객취소_집품후        525   476    0      0    0
과집              113    96    0      0    0
단순취소           4721     0    0      0    0
리빈파손            120     0   17      0  103
미집              547     0  104      0  443
오집             1163     0  874      0  289
집품파손             88     0   15      0   73
타토트오집_공급       1163   982    0      0    0
토트분실            223    53   16    138   69
자동화 당일 가용 공급원 1601 대체 1160 자기 오집 회수 실물로 대체 812
                 행수  등록검수    대체  원토트복구   확정
cancel_reason                              
고객취소_집품후        525   476     0      0    0
과집              113    97     0      0    0
단순취소           4721     0     0      0    0
리빈파손            120     0    17      0  103

## STEP 11-G — 동일 표본 실물만 vs 가상 요청에도 처리시간 부과
진단 모드만 가상 요청에 처리시간을 적용합니다.

In [34]:
# STEP 11-G (v13): 동일한 1:9 표본에서 가상 요청에 중복 서비스 시간을 부과한 경우와 실물 전용 큐를 비교한다.
# 처리능력은 기존 200/420개 실물/h 가정으로 고정. 진단용 중복 부하는 실제 운영 시나리오가 아니다.
def picking_diagnostic_row(share, capacity, bm, am):
    b=bm[bm['demand_flag'].eq(1)]; a=am[am['demand_flag'].eq(1)]
    bcan=int(b['op_cancel_confirmed_flag'].sum()); acan=int(a['op_cancel_confirmed_flag'].sum())
    return {'과집_비중_가정':share,'수동_시간당_처리량':capacity,'대체복구_수요':len(b),
        '수동_평균_지연분':b['process_delay_min'].mean(),'수동_재고대체율':b['reused_flag'].mean(),
        '자동_재고대체율':a['reused_flag'].mean(),'수동_확정':bcan,'자동_확정':acan,
        '취소_감소율':(bcan-acan)/bcan}

picking_diagnostic_rows=[picking_diagnostic_row(OVERPICK_SHARE_OF_WRONG_PICK,MANUAL_THROUGHPUT,final_df,match_input_df)]
picking_diagnostic_rows[0]['큐_구분']='실물만'
def duplicate_work_diagnostic(base,mode):
    x=build_scenario_pre_match(base,mode,1.0,include_virtual_work=True)
    return add_operational_outcomes(assign_reuse_with_time_window(add_physical_flags(x),mode))
duplicate_manual=duplicate_work_diagnostic(base_cancel_df,'manual')
duplicate_auto=duplicate_work_diagnostic(base_cancel_df,'automated')
row=picking_diagnostic_row(OVERPICK_SHARE_OF_WRONG_PICK,MANUAL_THROUGHPUT,duplicate_manual,duplicate_auto)
row['큐_구분']='실물+가상 요청 중복(진단용)'
picking_diagnostic_rows.append(row)
picking_diagnostic_df=pd.DataFrame(picking_diagnostic_rows)
print(picking_diagnostic_df.round(4).to_string(index=False))
for label,x in [('수동 실물만',final_df),('자동 실물만',match_input_df),('수동 중복',duplicate_manual),('자동 중복',duplicate_auto)]:
    physical=x.loc[x['physical_fifo_service_count'].gt(0)]
    print(label,'실물 슬롯',int(x['physical_fifo_service_count'].sum()),'가상 슬롯',int(x['virtual_fifo_service_count'].sum()),'평균 실물 대기(분)',physical['cancel_zone_queue_wait_min'].mean())


 과집_비중_가정  수동_시간당_처리량  대체복구_수요  수동_평균_지연분  수동_재고대체율  자동_재고대체율  수동_확정  자동_확정  취소_감소율             큐_구분
      0.1         200     2141    11.9142    0.4792    0.5418    977    826  0.1546              실물만
      0.1         200     2141   302.5592    0.1102    0.5418   1775    826  0.5346 실물+가상 요청 중복(진단용)
수동 실물만 실물 슬롯 2104 가상 슬롯 0 평균 실물 대기(분) 11.53615990774715
자동 실물만 실물 슬롯 2104 가상 슬롯 0 평균 실물 대기(분) 0.06384662164131812
수동 중복 실물 슬롯 2104 가상 슬롯 1803 평균 실물 대기(분) 301.1267040040082
자동 중복 실물 슬롯 2104 가상 슬롯 1778 평균 실물 대기(분) 0.2662063852423954
